# Per-iteration precision against the precision that saves a pathologist time: `candrefine_rerank_mo_pad61_select`

**What this notebook does.** It replays the AnnotateDx click-review loop over 6 iterations on the 49 ROIs. Each ROI's
6 recorded clicks are taken in 20 random orders. The loop uses the saved per-click candidate pools of
`NOTEBOOK = candrefine_rerank_mo_pad61_select.ipynb`. It never re-runs a search and reads no TIFF.

Each iteration, or **page**, works like this:
1. The pathologist boxes the next click.
2. The tool ranks that click's saved pool, minus everything already shown and every annotation already found.
3. It shows the top K.
4. The pathologist accepts the mitoses and rejects the rest.

Every page is then scored against the break-even precision `p*`, the precision at which reviewing the page finds
mitoses as fast as hand search does. The specification is `ITERATION_BREAKEVEN_PROMPT.md`.

**Arms simulated.**
- `C1` (= `REF`), NOTEBOOK's frozen choice. Its spec is `candrefine_rerank_spec.json`, and it is also the highest-P@10
  arm (checked below on integer TP).
- `od_unfiltered`, the user's current ordering: `chromatin_od` on the whole pool.
- `tm_unfiltered`, the D5 production default: `tm_score` on the whole pool.
- `od`, `chromatin_od` on NOTEBOOK's refusal-filtered pool.

Every arm runs as **recomputed**, the primary run: it ranks inside each remaining pool. Two further variants:
- **fullpool** (`C1` only) keeps the full-pool order and skips the removed items. A single-key arm's two orders are
  identical, and this is asserted on every page.
- **refill** (`od_unfiltered` and `tm_unfiltered`) refills each page's pool from the click's deep top-1000 list. This
  models a product that hides shown candidates before the 100-peak cap. The deep list is an input of NOTEBOOK, so the
  saved data *can* model refill for these two arms, contrary to the prompt.

**In-sample.** `C1`'s rule was found on these 49 ROIs, and NOTEBOOK chose it on these same 294 clicks. Every `C1`
number here is optimistic. The comparison arms carry no selection.

**Pinned choices the prompt leaves open:**
- **Orders.**
  - Each ROI's clicks are sorted by `seed_index`.
  - One generator per ROI, `np.random.default_rng([20261003, image_id])`, with `image_id` from NOTEBOOK's clicks CSV
    (asserted equal to the JSON's).
  - 20 successive `rng.permutation(6)` calls.
  - The same orders serve every arm and K.
- **Skipped page.** The pool is empty but r > 0. It costs `c + L` (a box and a wait), yields nothing, and counts as
  E = 0 in the all-pages shares. It is excluded from the reachable-only shares.
- **Session end.** It ends when r = 0 after the box. An ended session has no further rows.
- **Stop rules.**
  - The oracle stop excludes the first page with E < 1 and every page after it.
  - The observable stop includes the first zero-TP page, because its cost has been paid, and then stops.
- **Page 1 per ROI** is computed exactly, from every click's standalone first page (6 per ROI). The simulated
  iteration-1 mean weights clicks by how often each comes first in the 20 orders, so it only approximates NOTEBOOK's
  P@K.
- **Central `c_saved`** is 6 s, the middle of the grid. The prompt names no central value.

**Deviations, all stated again in the verdict:**
- `pages.csv` stores the inputs `n`, `TP`, `r` and `R0` and the central-setting columns only. Every grid value is a
  closed-form function of those inputs, and is recomputed in the grid cell instead of stored 9 times per page.
- Two accountings are added next to the prompt's per-page sum of seconds saved:
  - a session figure with r fixed at the ROI total;
  - that figure minus the extra hand-search cost of seeds in a depleted ROI.
- A density-dependent T (`T + r*c_saved`) is added as a sensitivity.
- A closed-form break-even pass time `T_break` per ROI is added. Above it, the tool saves time.
- The refill variant is added.

The cost-model cell explains why.

In [1]:
import hashlib
import inspect
import json
import os
import sys
import time
from fractions import Fraction

import numpy as np
import pandas as pd

sys.path.insert(0, '../..')
from midog_utils import dataset as ds
from midog_utils import evaluate as ev

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 600)
pd.set_option('display.max_colwidth', 120)

NB_T0 = time.time()

# NOTEBOOK: the notebook whose saved pools this one replays
NB_STEM = 'candrefine_rerank_mo_pad61_select'
NOTEBOOK_PATH = f'{NB_STEM}.ipynb'
SPEC_PATH = 'data/candrefine_rerank_spec.json'
SPEC_SHA1_RECORDED = '31392ada769cd37563454fb1f529ef660fc81154'   # the SHA-1 NOTEBOOK printed when it wrote the spec
DB_PATH = '../../databases/MIDOG++.json'
SRC = f'../../results/{NB_STEM}'
INPUTS = {name: f'{SRC}_{name}.csv' for name in ('per_run', 'pool', 'clicks', 'per_roi', 'delta_stats', 'combos', 'auc', 'refusals')}
INPUTS.update(spec=SPEC_PATH, notebook=NOTEBOOK_PATH, midog_json=DB_PATH, deep_top1000='../../results/deep_pool_mo_pad61_49roi_6seed_postD11_top1000.csv')
OUT_STEM = f'../../results/iteration_breakeven_{NB_STEM}'
OUT = {name: f'{OUT_STEM}_{name}.csv' for name in ('pages', 'per_roi', 'targets', 'grid', 'summary')}
assert not set(map(os.path.abspath, OUT.values())) & set(map(os.path.abspath, INPUTS.values())), 'an output path would overwrite an input'

# the simulation
KS = (10, 20, 30)
N_CLICKS_PER_ROI = 6
N_ORDERS = 20
N_ITER = 6
ORDER_SEED = 20261003
N_BOOT = 10_000
BOOT_SEED = 20260917
COMPARISON_ARMS = ('od_unfiltered', 'tm_unfiltered', 'od')
REFILL_ARMS = ('od_unfiltered', 'tm_unfiltered')   # the arms whose pools can be refilled from the deep list (no features needed)
PRODUCTION = 'tm_unfiltered'    # D5's default ranker on the whole pool
CURRENT = 'od_unfiltered'       # the user's current ordering

# the cost model, in seconds; none of these has been measured
V_GRID = (1.5, 2.0, 3.0)        # v: judge one candidate
T_GRID_MIN = (5, 10, 20)        # T: one complete manual pass over the ROI, minutes
C_BOX = 5.0                     # c: draw the box
L_WAIT = 2.0                    # L: latency
C_SAVED_GRID = (4.0, 6.0, 10.0) # c_saved: decide-and-mark time inside a mandatory pass (regime B, and the density-T sensitivity)
V0, T0_MIN, C_SAVED0 = 2.0, 10, 6.0
GRID = [(v, t) for v in V_GRID for t in T_GRID_MIN]
R_TABLE = (5, 10, 20, 40, 80, 160)
DENSITY_EDGES = (20, 40, 80)
DENSITY_LABELS = ('<=20', '21-40', '41-80', '>80')


def density_bin(R0):
    """
        The density bin of a ROI from its total mitoses.

        R0 (int): the ROI's category-1 count in the JSON.

        Returns str: one of DENSITY_LABELS.
    """
    for edge, lab in zip(DENSITY_EDGES, DENSITY_LABELS):
        if R0 <= edge:
            return lab
    return DENSITY_LABELS[-1]


print('outputs:', *OUT.values(), sep='\n  ')

outputs:
  ../../results/iteration_breakeven_candrefine_rerank_mo_pad61_select_pages.csv
  ../../results/iteration_breakeven_candrefine_rerank_mo_pad61_select_per_roi.csv
  ../../results/iteration_breakeven_candrefine_rerank_mo_pad61_select_targets.csv
  ../../results/iteration_breakeven_candrefine_rerank_mo_pad61_select_grid.csv
  ../../results/iteration_breakeven_candrefine_rerank_mo_pad61_select_summary.csv


In [2]:
def sha256_12(path):
    """
        The first 12 hex digits of a file's SHA-256, so the inputs this notebook read are on record.

        path (str): the file to hash.

        Returns str: 12 hex characters.
    """
    with open(path, 'rb') as fh:
        return hashlib.sha256(fh.read()).hexdigest()[:12]


INPUT_SHA = {name: sha256_12(p) for name, p in INPUTS.items()}
for name, p in INPUTS.items():
    print(f'  {INPUT_SHA[name]}  {p}')
with open(SPEC_PATH, 'rb') as fh:
    spec_bytes = fh.read()
assert hashlib.sha1(spec_bytes).hexdigest() == SPEC_SHA1_RECORDED, 'the spec file is not the one NOTEBOOK wrote'
SPEC = json.loads(spec_bytes)
assert not SPEC['smoke'] and SPEC['cue']['combination'] == 'C1' and SPEC['cue']['is_ref'] and SPEC['ref']['combination'] == 'C1', 'the spec does not freeze C1 = REF'
ROI_ORDER = list(SPEC['rois'])
F5_DIR = int(SPEC['f5_direction_all_rois'])
assert len(ROI_ORDER) == 49 and F5_DIR in (1, -1)

NBCSV = {name: pd.read_csv(INPUTS[name], float_precision='round_trip') for name in ('per_run', 'pool', 'clicks', 'per_roi', 'delta_stats', 'combos', 'auc', 'refusals')}
PER_RUN, POOL, CLICKS = NBCSV['per_run'], NBCSV['pool'], NBCSV['clicks']
assert (len(PER_RUN), len(POOL), len(CLICKS)) == (3234, 29107, 294), 'a NOTEBOOK output does not have its expected row count'
assert set(CLICKS['file_name']) == set(ROI_ORDER) and set(PER_RUN['file_name']) == set(ROI_ORDER) and set(POOL['file_name']) == set(ROI_ORDER)
DEEP = pd.read_csv(INPUTS['deep_top1000'], float_precision='round_trip')
assert len(DEEP) == 294000 and DEEP[['cx', 'cy', 'score', 'od']].notna().all().all(), 'the deep list is not 1000 complete rows per click'
images, annotations = ds.load_annotations(DB_PATH)
DOMAIN_OF = CLICKS.drop_duplicates('file_name').set_index('file_name')['domain']
print(f"\nspec: cue {SPEC['cue']['combination']} (= REF), terms {'+'.join(SPEC['cue']['terms'])}; F5 direction {F5_DIR:+d}; {len(ROI_ORDER)} ROIs; fold agreement {SPEC['fold_agreement']}/{len(SPEC['folds'])}")

  1fcc9dd4120b  ../../results/candrefine_rerank_mo_pad61_select_per_run.csv
  e5b38df77cc6  ../../results/candrefine_rerank_mo_pad61_select_pool.csv
  bcca7f80914e  ../../results/candrefine_rerank_mo_pad61_select_clicks.csv
  f8b2407cd7d7  ../../results/candrefine_rerank_mo_pad61_select_per_roi.csv
  4204f81abcea  ../../results/candrefine_rerank_mo_pad61_select_delta_stats.csv
  9e33cb46d767  ../../results/candrefine_rerank_mo_pad61_select_combos.csv
  11701d4c6af6  ../../results/candrefine_rerank_mo_pad61_select_auc.csv
  ef4e48aa1743  ../../results/candrefine_rerank_mo_pad61_select_refusals.csv
  648a9807cf4b  candrefine_rerank_spec.json
  2f507898331c  candrefine_rerank_mo_pad61_select.ipynb
  8c87ab4276eb  ../../databases/MIDOG++.json
  5b847bd9cf92  ../../results/deep_pool_mo_pad61_49roi_6seed_postD11_top1000.csv



spec: cue C1 (= REF), terms F0+F1+F2+F3+T; F5 direction -1; 49 ROIs; fold agreement 7/7


## NOTEBOOK's ranking functions, verbatim

`rank_by`, `term_ranks`, `combo_order` and `arm_lists` are copied byte for byte from NOTEBOOK. Their source SHA-1s are
asserted against the spec's `function_sha1`. `auc_u`, `direction_from_auc` and `pick_combo` are also copied verbatim,
so that NOTEBOOK's leave-one-domain-out choice can be recomputed below. The spec does not hash those three.

In [3]:
# copied verbatim from cell 1 of candrefine_rerank_mo_pad61_select.ipynb
TERMS = ('F0', 'T', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9')
COLUMN = {'F0': 'od', 'T': 'tm_rank', 'F1': 'core_od', 'F2': 'not_white', 'F3': 'solidity', 'F4': 'condensed', 'F5': 'texture', 'F6': 'rim_core', 'F7': 'spiky', 'F8': 'e_over_h', 'F9': 'd_over_h'}
DIRECTION = {'F0': 1, 'T': -1, 'F1': 1, 'F2': 1, 'F3': -1, 'F4': 1, 'F5': None, 'F6': -1, 'F7': 1, 'F8': -1, 'F9': -1}   # +1: larger is more mitosis-like; F5 is set per fold
REF_TERMS = ('F0', 'F1', 'F2', 'F3', 'T')
COMBOS = {'C1': REF_TERMS, 'C2': REF_TERMS + ('F8', 'F9'), 'C3': REF_TERMS + ('F4', 'F5'), 'C4': REF_TERMS + ('F6', 'F7'), 'C5': REF_TERMS + ('F4', 'F5', 'F6', 'F7', 'F8', 'F9')}
SELECT_TOLERANCE = Fraction(5, 1000)   # 0.5 points of (P@10 + P@20) / 2
AXIS_KEY = {'chromatin_od': 'od', 'tm_score': 'score'}
assert COLUMN == SPEC['term_columns'] and {c: list(t) for c, t in COMBOS.items()} == SPEC['combinations'] and {t: (F5_DIR if t == 'F5' else d) for t, d in DIRECTION.items()} == SPEC['directions'], 'the copied constants differ from the spec'
assert list(COMBOS['C1']) == SPEC['cue']['terms'] and SPEC['tie_break'] == "od rank (F0), then tm_rank, kind='mergesort'"


def rank_by(det, rank_key):
    """
        One ranking axis applied to a detection list, in run_production_pipeline's own sort order.

        det (pd.DataFrame): the unranked detections from search_once.
        rank_key (str): "chromatin_od" or "tm_score".

        Returns pd.DataFrame: the detections ranked best-first, with rank reassigned.
    """
    out = det.sort_values(AXIS_KEY[rank_key], ascending=False, na_position='last', kind='mergesort').reset_index(drop=True)
    return out.assign(rank=np.arange(len(out)))


def term_ranks(pool, f5_direction):
    """
        Every term's within-pool rank: 1 = most mitosis-like, average ties, NaN ranked worst.

        pool (pd.DataFrame): one click's accepted pool, with one column per term (COLUMN).
        f5_direction (int): +1 or -1, F5's direction.

        Returns pd.DataFrame: one rank column per term id, on the pool's index.
    """
    return pd.DataFrame({t: pool[COLUMN[t]].rank(method='average', na_option='bottom', ascending=(f5_direction if t == 'F5' else DIRECTION[t]) < 0) for t in TERMS}, index=pool.index)


def combo_order(pool, ranks, terms):
    """
        The pool sorted best-first by the equal-weight mean of its terms' ranks, ties broken by od rank and then tm_rank.

        pool (pd.DataFrame): one click's accepted pool.
        ranks (pd.DataFrame): term_ranks of the same pool.
        terms (tuple[str]): the term ids to average.

        Returns pd.DataFrame: the pool reordered, with rank reassigned 0..n-1.
    """
    key = pd.DataFrame({'mean_rank': ranks[list(terms)].mean(axis=1), 'od_rank': ranks['F0'], 'tm_rank': pool['tm_rank']})
    out = pool.loc[key.sort_values(['mean_rank', 'od_rank', 'tm_rank'], kind='mergesort').index].reset_index(drop=True)
    return out.assign(rank=np.arange(len(out)))


def arm_lists(pool):
    """
        Every arm's ranking of one click's accepted pool: od, tm and the five combinations, C3 and C5 under both F5 directions.

        pool (pd.DataFrame): the accepted pool in tm_score order, with tm_rank and one column per term.

        Returns dict: (arm, f5_direction) -> the ranked pool; f5_direction is 0 where F5 is not a term.
    """
    lists = {('od', 0): rank_by(pool, 'chromatin_od'), ('tm', 0): rank_by(pool, 'tm_score')}
    ranks = {d: term_ranks(pool, d) for d in (1, -1)}
    for c, terms in COMBOS.items():
        for d in ((1, -1) if 'F5' in terms else (0,)):
            lists[(c, d)] = combo_order(pool, ranks[d if d else 1], terms)
    return lists


def auc_u(values, is_mitosis):
    """
        The Mann-Whitney AUC of one feature, NaN values left out: the chance a mitosis has the larger value, ties counted one half.

        values (array-like): one value per (click, candidate) row.
        is_mitosis (array-like): True for a mitosis.

        Returns tuple[float, int, int, int]: the AUC (NaN when a class is empty), 2U as an exact integer, and the mitosis and other counts.
    """
    v = pd.Series(np.asarray(values, dtype=float))
    y = np.asarray(is_mitosis, dtype=bool)
    ok = v.notna().to_numpy()
    v, y = v[ok].reset_index(drop=True), y[ok]
    n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0:
        return np.nan, 0, n1, n0
    two_u = int(round(2 * v.rank(method='average').to_numpy()[y].sum())) - n1 * (n1 + 1)
    return two_u / (2 * n1 * n0), two_u, n1, n0


def direction_from_auc(two_u, n1, n0):
    """
        A feature's direction from the sign of its AUC - 0.5, decided exactly; an AUC of exactly 0.5 gives +1.

        two_u (int): twice the Mann-Whitney U of the mitoses.
        n1 (int): the number of mitoses.
        n0 (int): the number of others.

        Returns int: +1 when larger is more mitosis-like, else -1.
    """
    assert n1 > 0 and n0 > 0, 'a direction needs both classes'
    return 1 if two_u >= n1 * n0 else -1


def pick_combo(score):
    """
        The selection rule: within SELECT_TOLERANCE of the best mean, the fewest terms, then the higher mean, then the lower C number.

        score (dict): combination -> its exact mean over ROIs of (P@10 + P@20) / 2, as a Fraction.

        Returns str: the picked combination.
    """
    best = max(score.values())
    near = [c for c in COMBOS if best - score[c] <= SELECT_TOLERANCE]
    fewest = min(len(COMBOS[c]) for c in near)
    tied = [c for c in near if len(COMBOS[c]) == fewest]
    top = max(score[c] for c in tied)
    return min(c for c in tied if score[c] == top)


COPIED_SHA1 = {f.__name__: hashlib.sha1(inspect.getsource(f).encode()).hexdigest() for f in (rank_by, term_ranks, combo_order, arm_lists)}
for name, h in COPIED_SHA1.items():
    print(f'  {h}  {name}  (spec {SPEC["function_sha1"][name]})')
assert all(h == SPEC['function_sha1'][name] for name, h in COPIED_SHA1.items()), 'a copied ranking function is not byte-identical to NOTEBOOK\'s'
print('the copied ranking functions are byte-identical to NOTEBOOK\'s (source SHA-1 = spec)')

  b5f991248cd103a1e022a0b8bb76ec2a9c0230f6  rank_by  (spec b5f991248cd103a1e022a0b8bb76ec2a9c0230f6)
  0c7da09185104d1ae0d65d35b757f08f5651fb02  term_ranks  (spec 0c7da09185104d1ae0d65d35b757f08f5651fb02)
  9246668af589d2229bc665c5efd54fde85e6b20d  combo_order  (spec 9246668af589d2229bc665c5efd54fde85e6b20d)
  cbd73590750794ea0673f73b31040c249658889b  arm_lists  (spec cbd73590750794ea0673f73b31040c249658889b)
the copied ranking functions are byte-identical to NOTEBOOK's (source SHA-1 = spec)


## Statistics helpers

**The unit is the ROI.** A ROI's value is the mean over its pages in a cell, which averages over the 20 orders
first. The table value is the mean over ROIs. Its CI is a ROI-cluster bootstrap: `N_BOOT` 10 000 resamples of the
ROIs, seed 20260917, percentile 95 %. A ROI with no page in a cell is left out of that cell, so every table reports
`n_roi`.

In [4]:
BOOT_IDX = {}


def boot_ci(values):
    """
        The mean of per-ROI values and its ROI-cluster bootstrap 95% CI.

        values (array-like): one value per ROI; NaN values are dropped.

        Returns tuple[float, float, float, int]: mean, CI low, CI high, and the number of ROIs.
    """
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    n = len(v)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    if n not in BOOT_IDX:
        BOOT_IDX[n] = np.random.default_rng(BOOT_SEED).integers(0, n, size=(N_BOOT, n))
    means = v[BOOT_IDX[n]].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(v.mean()), float(lo), float(hi), n


def summarise(frame, col, by):
    """
        The mean over ROIs of a per-page column, each ROI first averaged over its pages in the group, with a ROI-cluster bootstrap CI.

        frame (pd.DataFrame): page rows with file_name and the group columns.
        col (str): the column to average; NaN rows are left out.
        by (list[str]): the group columns.

        Returns pd.DataFrame: the group columns, then mean, ci_lo, ci_hi and n_roi.
    """
    per_roi = frame.groupby(by + ['file_name'], sort=False, observed=True)[col].mean()
    rows = []
    for key, g in per_roi.groupby(level=list(range(len(by))), sort=False):
        key = key if isinstance(key, tuple) else (key,)
        m, lo, hi, n = boot_ci(g.to_numpy())
        rows.append(dict(zip(by, key), mean=m, ci_lo=lo, ci_hi=hi, n_roi=n))
    return pd.DataFrame(rows)


def paired(a, b):
    """
        A paired ROI-level contrast: the mean of a - b over ROIs with its bootstrap CI, W/L/T and the median difference.

        a (pd.Series): per-ROI values of the first arm, indexed by file_name.
        b (pd.Series): per-ROI values of the second arm, indexed by file_name.

        Returns dict: mean, ci_lo, ci_hi, n_roi, wins, losses, ties, median.
    """
    d = (a - b.reindex(a.index)).dropna()
    m, lo, hi, n = boot_ci(d.to_numpy())
    return dict(mean=m, ci_lo=lo, ci_hi=hi, n_roi=n, wins=int((d > 0).sum()), losses=int((d < 0).sum()), ties=int((d == 0).sum()), median=float(d.median()))


def fmt_ci(m, lo, hi, scale=1.0, nd=3):
    """
        A mean and its CI as one printable string.

        m (float): the mean.
        lo (float): the CI's lower end.
        hi (float): the CI's upper end.
        scale (float): a multiplier, e.g. 100 for points.
        nd (int): decimals.

        Returns str: 'm [lo, hi]'.
    """
    return f'{scale * m:.{nd}f} [{scale * lo:.{nd}f}, {scale * hi:.{nd}f}]'


SUMMARY = []


def collect(table, metric, frame):
    """
        Append one summarised table to the long summary output.

        table (str): the table's name.
        metric (str): what the mean column measures.
        frame (pd.DataFrame): a summarise or paired result, with its group columns.

        Returns None: SUMMARY grows.
    """
    SUMMARY.append(frame.assign(table=table, metric=metric))

## Adversarial check 2: NOTEBOOK's own numbers, recomputed from its CSVs

The following are recomputed from NOTEBOOK's CSVs and compared with its outputs:
- every arm's pooled P@K;
- the best arm at each K, by exact integer TP. Ties are listed and never broken by column order;
- `per_roi`, `delta_stats` and the spec's `all_roi_scores`;
- the leave-one-domain-out picks, F5's direction included;
- the pooled per-feature AUCs;
- the refusal table.

The page-1 contrasts against the comparison arms then get a ROI-cluster bootstrap CI. NOTEBOOK reports none.

In [5]:
KCOLS = [f'tp_at_{k}' for k in KS]
N_CLICKS = len(ROI_ORDER) * N_CLICKS_PER_ROI
ARM_KEYS_NB = list(dict.fromkeys(zip(PER_RUN['arm'], PER_RUN['f5_direction'])))
TP_SUM = PER_RUN.groupby(['arm', 'f5_direction'], sort=False)[KCOLS].sum()
assert (PER_RUN.groupby(['arm', 'f5_direction'], sort=False).size() == N_CLICKS).all()


def arm_name(arm, d):
    """
        A printable arm name, with F5's direction where F5 is a term.

        arm (str): the arm.
        d (int): its F5 direction, 0 when F5 is not a term.

        Returns str: e.g. 'C3 (F5 -1)'.
    """
    return arm if d == 0 else f'{arm} (F5 {d:+d})'


PREC_NB = pd.DataFrame([dict(arm=arm_name(*key), **{f'TP@{k}': int(TP_SUM.loc[key, f'tp_at_{k}']) for k in KS}, **{f'P@{k}': TP_SUM.loc[key, f'tp_at_{k}'] / (k * N_CLICKS) for k in KS}) for key in ARM_KEYS_NB])
print('pooled precision of every NOTEBOOK arm, recomputed from per_run (294 clicks; the denominator is K x 294):')
print(PREC_NB.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
QUOTED_P10 = {'C1': 0.637, 'od': 0.589, 'od_unfiltered': 0.563, 'tm_unfiltered': 0.542}
for arm, q in QUOTED_P10.items():
    assert round(TP_SUM.loc[(arm, 0), 'tp_at_10'] / (10 * N_CLICKS), 3) == q, f'{arm}: pooled P@10 is not the quoted {q}'
print('the quoted pooled P@10 values (C1 0.637, od 0.589, od_unfiltered 0.563, tm_unfiltered 0.542) reproduce')

# the best arm at each K, by exact integer TP; every arm at the spec's F5 direction (the other direction is not a candidate)
LEGIT = [key for key in ARM_KEYS_NB if key[1] in (0, F5_DIR)]
BEST_BY_K = {}
print('\narms ranked by total TP at each K (exact integers; ties listed, never broken by order):')
for k in KS:
    tps = sorted(((int(TP_SUM.loc[key, f'tp_at_{k}']), key) for key in LEGIT), key=lambda t: -t[0])
    top = tps[0][0]
    BEST_BY_K[k] = [key for v, key in tps if v == top]
    runner = next(v for v, _ in tps if v < top)
    print(f'  K={k}: ' + ', '.join(f'{arm_name(*key)} {v}' for v, key in tps) + f'  -> leader(s) {[arm_name(*key) for key in BEST_BY_K[k]]}, {top - runner} TP ahead of the next ({100 * (top - runner) / (k * N_CLICKS):.2f} pts)')
BEST = SPEC['cue']['combination']
SIM_BEST = [BEST] + [key[0] for key in BEST_BY_K[10] if key[0] != BEST]
assert all(key[1] == 0 for key in BEST_BY_K[10]), 'a K=10 leader has F5 as a term: the simulator would need its direction'
SIM_ARMS = SIM_BEST + list(COMPARISON_ARMS)
print(f'NOTEBOOK\'s pre-declared choice: {BEST}. Highest pooled P@10: {[arm_name(*key) for key in BEST_BY_K[10]]}. Simulated arms: {SIM_ARMS}')

# per_roi.csv is per_run summed by ROI
pr_sum = PER_RUN.groupby(['file_name', 'arm', 'f5_direction'])[KCOLS].sum().sort_index()
nb_roi = NBCSV['per_roi'].set_index(['file_name', 'arm', 'f5_direction'])[KCOLS].sort_index()
assert nb_roi.index.equals(pr_sum.index) and (nb_roi.to_numpy() == pr_sum.to_numpy()).all(), 'per_roi.csv is not per_run summed by ROI'
print('per_roi.csv = per_run summed by ROI: OK')

ROI_TP = PER_RUN.groupby(['arm', 'f5_direction', 'file_name'])[KCOLS].sum()


def roi_tp(key, k):
    """
        Per-ROI TP@K of one NOTEBOOK arm, in ROI_ORDER.

        key (tuple[str, int]): the (arm, f5_direction) key.
        k (int): the budget.

        Returns np.ndarray: TP@K summed over each ROI's 6 clicks.
    """
    return ROI_TP.loc[key][f'tp_at_{k}'].reindex(ROI_ORDER).to_numpy()


def parse_arm(label):
    """
        The (arm, f5_direction) key of a NOTEBOOK contrast operand.

        label (str): e.g. 'REF', 'od_unfiltered' or 'C3 (F5 -1)'.

        Returns tuple[str, int]: the per_run key.
    """
    if label == 'REF':
        return ('C1', 0)
    if label.endswith(')'):
        arm, d = label[:-1].split(' (F5 ')
        return (arm, int(d))
    return (label, 0)


dchk = []
for r in NBCSV['delta_stats'].itertuples():
    a, b = (parse_arm(x) for x in r.contrast.split(' - '))
    d = roi_tp(a, r.K) - roi_tp(b, r.K)
    dchk.append(dict(contrast=r.contrast, K=r.K, nb_pts=r.pooled_delta_pts, re_pts=100 * d.sum() / (r.K * N_CLICKS), nb_wlt=(r.wins, r.losses, r.ties), re_wlt=(int((d > 0).sum()), int((d < 0).sum()), int((d == 0).sum()))))
DCHK = pd.DataFrame(dchk)
DCHK['match'] = np.isclose(DCHK['nb_pts'], DCHK['re_pts'], rtol=0, atol=1e-9) & (DCHK['nb_wlt'] == DCHK['re_wlt'])
print(f"delta_stats.csv: {int(DCHK['match'].sum())}/{len(DCHK)} rows reproduce (pooled delta and W/L/T)")
assert DCHK['match'].all(), DCHK[~DCHK['match']]


def roi_score_exact(key, f):
    """
        One ROI's (P@10 + P@20) / 2 for one NOTEBOOK arm, exactly.

        key (tuple[str, int]): the (arm, f5_direction) key.
        f (str): the ROI.

        Returns Fraction: the score over its 6 clicks.
    """
    g = ROI_TP.loc[key + (f,)]
    return (Fraction(int(g['tp_at_10']), 10 * N_CLICKS_PER_ROI) + Fraction(int(g['tp_at_20']), 20 * N_CLICKS_PER_ROI)) / 2


def combo_key(c, d):
    """
        The per_run key of a combination at a given F5 direction.

        c (str): the combination.
        d (int): F5's direction.

        Returns tuple[str, int]: (c, d) when F5 is a term, else (c, 0).
    """
    return (c, d if 'F5' in COMBOS[c] else 0)


SCORE_RE = {c: sum((roi_score_exact(combo_key(c, F5_DIR), f) for f in ROI_ORDER), Fraction(0)) / len(ROI_ORDER) for c in COMBOS}
assert all(SCORE_RE[c] == Fraction(SPEC['all_roi_scores'][c]) for c in COMBOS), 'the spec all_roi_scores do not reproduce'
print('spec all_roi_scores reproduce exactly: ' + ', '.join(f'{c} {v} = {float(v):.4f}' for c, v in SCORE_RE.items()))

LAB = POOL[POOL['accepted']]
assert (LAB['is_mitosis'] == (LAB['label_bucket'] == ev.HUMAN_CORRECT_LABEL)).all()
a_all, two_u, n1, n0 = auc_u(LAB['texture'], LAB['is_mitosis'])
assert np.isclose(a_all, SPEC['f5_auc_all_rois'], rtol=0, atol=1e-15) and direction_from_auc(two_u, n1, n0) == F5_DIR
fold_rows = []
for fold in SPEC['folds']:
    dom = fold['held_out_domain']
    train = [f for f in ROI_ORDER if DOMAIN_OF[f] != dom]
    held = [f for f in ROI_ORDER if DOMAIN_OF[f] == dom]
    sub = LAB[LAB['file_name'].isin(train)]
    _, tu, m1, m0 = auc_u(sub['texture'], sub['is_mitosis'])
    d = direction_from_auc(tu, m1, m0)
    score = {c: sum((roi_score_exact(combo_key(c, d), f) for f in train), Fraction(0)) / len(train) for c in COMBOS}
    pick = pick_combo(score)
    held_score = {c: sum((roi_score_exact(combo_key(c, d), f) for f in held), Fraction(0)) / len(held) for c in COMBOS}
    held_od = sum((roi_score_exact(('od', 0), f) for f in held), Fraction(0)) / len(held)
    fold_rows.append(dict(held_out=dom, f5=d, pick=pick, spec_pick=fold['pick'], held_delta_vs_ref=held_score[pick] - held_score['C1'], spec_delta=Fraction(fold['held_out_delta_vs_ref']), held_C1_minus_od_pts=100 * float(held_score['C1'] - held_od)))
FOLD_RE = pd.DataFrame(fold_rows)
assert (FOLD_RE['pick'] == FOLD_RE['spec_pick']).all() and (FOLD_RE['held_delta_vs_ref'] == FOLD_RE['spec_delta']).all() and (FOLD_RE['f5'] == [f['f5_direction'] for f in SPEC['folds']]).all()
print('\nleave-one-domain-out recomputed from the pool CSV and per_run: every fold\'s F5 direction, pick and held-out delta reproduce')
print(FOLD_RE[['held_out', 'f5', 'pick', 'held_delta_vs_ref', 'held_C1_minus_od_pts']].to_string(index=False, float_format=lambda v: f'{v:+.2f}'))
print('-> every fold picked C1 = REF, so every "held-out delta vs REF" is 0 by construction. It is not held-out evidence that C1 beats od; the C1 - od column is in-sample (C1\'s terms were chosen on these ROIs before NOTEBOOK ran).')

auc_nb = NBCSV['auc'].query("scope == 'pooled'").set_index('term')['auc_raw']
for t in TERMS:
    assert np.isclose(auc_u(LAB[COLUMN[t]], LAB['is_mitosis'])[0], auc_nb[t], rtol=0, atol=1e-12), f'pooled AUC of {t} does not reproduce'
print('pooled per-feature AUCs reproduce from the pool CSV')

BUCKET_SHORT = {ev.HUMAN_CORRECT_LABEL: 'mitosis', ev.HUMAN_REJECTED_LABEL: 'look-alike', ev.NON_HUMAN_FINDINGS: 'unlabelled'}
ref_re = POOL.assign(b=POOL['bucket_unfiltered'].map(BUCKET_SHORT)).groupby(['refusal', 'b']).size().unstack(fill_value=0)
ref_nb = NBCSV['refusals'].set_index('refusal')
assert all(int(ref_re.loc[c].get(b, 0)) == int(ref_nb.loc[c, b]) for c in ref_re.index for b in BUCKET_SHORT.values()), 'the refusal table does not reproduce'
print('refusal table reproduces from the pool CSV')

# page-1 contrasts with a ROI-cluster bootstrap (NOTEBOOK reports only pooled deltas and W/L/T)
print('\npage-1 (iteration-1) contrasts, ROI-level pooled-precision difference in points, ROI-cluster bootstrap 95% CI (in-sample for C1):')
P1_CONTRASTS = []
for other in COMPARISON_ARMS:
    for k in KS:
        d = (roi_tp(('C1', 0), k) - roi_tp((other, 0), k)) / (k * N_CLICKS_PER_ROI)
        m, lo, hi, n = boot_ci(d)
        P1_CONTRASTS.append(dict(contrast=f'C1 - {other}', K=k, delta_pts=100 * m, ci_lo=100 * lo, ci_hi=100 * hi, wins=int((d > 0).sum()), losses=int((d < 0).sum()), ties=int((d == 0).sum())))
P1_CONTRASTS = pd.DataFrame(P1_CONTRASTS)
print(P1_CONTRASTS.to_string(index=False, float_format=lambda v: f'{v:+.2f}'))
collect('page1_contrast', 'C1 minus arm, pooled precision points', P1_CONTRASTS.rename(columns={'delta_pts': 'mean'}))

pooled precision of every NOTEBOOK arm, recomputed from per_run (294 clicks; the denominator is K x 294):
          arm  TP@10  TP@20  TP@30   P@10   P@20   P@30
od_unfiltered   1654   2993   3907 0.5626 0.5090 0.4430
tm_unfiltered   1593   2755   3671 0.5418 0.4685 0.4162
           od   1731   3054   3989 0.5888 0.5194 0.4523
           tm   1620   2787   3703 0.5510 0.4740 0.4198
           C1   1874   3231   4253 0.6374 0.5495 0.4822
           C2   1796   3089   4079 0.6109 0.5253 0.4625
   C3 (F5 +1)   1786   3122   4131 0.6075 0.5310 0.4684
   C3 (F5 -1)   1834   3176   4128 0.6238 0.5401 0.4680
           C4   1835   3195   4172 0.6241 0.5434 0.4730
   C5 (F5 +1)   1678   2920   3929 0.5707 0.4966 0.4455
   C5 (F5 -1)   1816   3069   4036 0.6177 0.5219 0.4576
the quoted pooled P@10 values (C1 0.637, od 0.589, od_unfiltered 0.563, tm_unfiltered 0.542) reproduce

arms ranked by total TP at each K (exact integers; ties listed, never broken by order):
  K=10: C1 1874, C4 1835, C3 (

spec all_roi_scores reproduce exactly: C1 997/1680 = 0.5935, C2 2227/3920 = 0.5681, C3 1711/2940 = 0.5820, C4 1373/2352 = 0.5838, C5 6701/11760 = 0.5698



leave-one-domain-out recomputed from the pool CSV and per_run: every fold's F5 direction, pick and held-out delta reproduce
                        held_out  f5 pick held_delta_vs_ref  held_C1_minus_od_pts
canine cutaneous mast cell tumor  -1   C1                 0                 -1.13
              canine lung cancer  -1   C1                 0                 +0.00
            canine lymphosarcoma  -1   C1                 0                 +5.71
      canine soft tissue sarcoma  -1   C1                 0                 +1.43
             human breast cancer  -1   C1                 0                 +6.67
                  human melanoma  -1   C1                 0                 +9.29
      human neuroendocrine tumor  -1   C1                 0                 +5.60
-> every fold picked C1 = REF, so every "held-out delta vs REF" is 0 by construction. It is not held-out evidence that C1 beats od; the C1 - od column is in-sample (C1's terms were chosen on these ROIs before NOTEBOOK r

## The simulator

`simulate_session` replays one session: one ROI, one click order, one arm, one K and one ranking variant.

Each iteration `j` with click `s`:
1. **Box.** `s`'s annotation is added to `found`, and the notebook notes whether it was already there. The unfound
   count `r` then excludes it.
2. **Pool.** The pool is `s`'s saved pool: accepted rows for filtered arms, all rows for unfiltered arms. Every
   candidate within the match radius (inclusive) of either of these is removed:
   - an earlier shown candidate;
   - any `found` annotation other than `s`'s own.
3. **Rank.**
   - `recomputed`: the arm's own function on the remaining pool. A combination recomputes its within-pool ranks.
   - `fullpool`: the full-pool order with removed items skipped.
   - For a single-key arm the two are asserted identical on every page.
   - `refill`: the same removals are applied to the click's deep list. The first `n_pool` survivors are kept, in
     score order, and then ranked.
4. **Page.** The page is the first `min(K, remaining)` candidates.
   - If the pool is empty, the page is skipped (`n = 0`).
   - If `r = 0`, the session ends.
5. **Match.** `ev.bucket_detections` matches the page against ground truth (both categories) minus `found` minus
   `claimed`. The matched annotations are claimed, and the matched mitoses are found.

**Diagnostics per page:**
- `mitoses_reachable`: unfound, unclaimed mitoses within R of at least one remaining candidate. This is an upper
  bound on what any ordering of the remaining pool could find.
- `hidden_mitoses`: unfound mitoses within R of a removed candidate and of no remaining one. The exclusions made
  these unreachable on this page.
- `hidden_near_shown`: those of them that also lie within R of an earlier shown candidate.

**Invariants, asserted on every page:**
- `found` only grows;
- no matched annotation was already found or claimed, and none repeats;
- `TP <= r`;
- no shown candidate lies within R of an earlier page's candidate.

In [6]:
HCL, HRL, NHF = ev.HUMAN_CORRECT_LABEL, ev.HUMAN_REJECTED_LABEL, ev.NON_HUMAN_FINDINGS
ARM_AXIS = {'od': 'chromatin_od', 'tm': 'tm_score', 'od_unfiltered': 'chromatin_od', 'tm_unfiltered': 'tm_score'}
KEEP_COLS = ['cx', 'cy', 'score', 'od', 'tm_rank', 'accepted'] + [COLUMN[t] for t in TERMS if COLUMN[t] not in ('od', 'tm_rank')]


def arm_base(arm):
    """
        Which pool an arm ranks.

        arm (str): the arm.

        Returns str: 'all' for the unfiltered arms, else 'accepted'.
    """
    return 'all' if arm.endswith('_unfiltered') else 'accepted'


def variants_for(arm):
    """
        The variants simulated for an arm: a combination adds the full-pool order with skips; an unfiltered arm adds the refill from the deep list.

        arm (str): the arm.

        Returns tuple[str]: ('recomputed', 'fullpool'), ('recomputed', 'refill') or ('recomputed',).
    """
    if arm in COMBOS:
        return ('recomputed', 'fullpool')
    return ('recomputed', 'refill') if arm in REFILL_ARMS else ('recomputed',)


def rank_remaining(arm, rem):
    """
        One arm's order of a pool, computed inside that pool with NOTEBOOK's own functions.

        arm (str): the arm.
        rem (pd.DataFrame): the candidates to rank, in tm_rank order, with a positional 'row' column.

        Returns np.ndarray: the 'row' values, best first.
    """
    if arm in COMBOS:
        return combo_order(rem, term_ranks(rem, F5_DIR), COMBOS[arm])['row'].to_numpy()
    return rank_by(rem, ARM_AXIS[arm])['row'].to_numpy()


def within(a, b, radius):
    """
        Which points of a lie within radius (inclusive) of at least one point of b.

        a (np.ndarray): (n, 2) points.
        b (np.ndarray): (m, 2) points.
        radius (float): the distance in pixels.

        Returns np.ndarray: n booleans.
    """
    if len(a) == 0 or len(b) == 0:
        return np.zeros(len(a), dtype=bool)
    d2 = ((a[:, None, :] - b[None, :, :]) ** 2).sum(axis=2)
    return (d2 <= radius ** 2).any(axis=1)


def prepare_click(frame, seed_index, seed_ann_id, arms, deep=None):
    """
        One click's pools, ready to simulate: every candidate and the accepted ones, each in tm_rank order, with each arm's full-pool order, and optionally the click's deep list for the refill variant.

        frame (pd.DataFrame): the click's rows of the pool CSV, in any order.
        seed_index (int): the click's seed index.
        seed_ann_id (int): the clicked annotation.
        arms (list[str]): the arms to prepare full-pool orders for.
        deep (pd.DataFrame or None): the click's deep detection list (cx, cy, score, od, rank), whose first n_pool rows must be the pool.

        Returns dict: seed_index, seed_ann_id, n_pool, base ('all' and 'accepted' frames with a positional 'row' column), xy (their coordinates), full_order (arm -> rows best first), and deep / deep_xy when given.
    """
    allf = frame.sort_values('tm_rank', kind='mergesort')[KEEP_COLS].reset_index(drop=True)
    assert (allf['tm_rank'].to_numpy() == np.arange(len(allf))).all(), 'tm_rank is not 0..n-1'
    allf['row'] = np.arange(len(allf))
    acc = allf[allf['accepted']].reset_index(drop=True)
    acc['row'] = np.arange(len(acc))
    base = {'all': allf, 'accepted': acc}
    out = dict(seed_index=int(seed_index), seed_ann_id=int(seed_ann_id), base=base, xy={b: f[['cx', 'cy']].to_numpy(dtype=float) for b, f in base.items()})
    out['full_order'] = {arm: rank_remaining(arm, base[arm_base(arm)]) for arm in arms}
    out['n_pool'] = len(allf)
    if deep is not None:
        dp = deep.sort_values('rank', kind='mergesort')[['cx', 'cy', 'score', 'od']].reset_index(drop=True)
        assert (np.diff(dp['score'].to_numpy()) <= 0).all() and len(dp) >= len(allf), 'the deep list is not in descending score order'
        assert np.array_equal(dp[['cx', 'cy', 'score', 'od']].to_numpy()[:len(allf)], allf[['cx', 'cy', 'score', 'od']].to_numpy()), 'the deep list does not begin with the saved pool'
        dp['row'] = np.arange(len(dp))
        out['deep'], out['deep_xy'] = dp, dp[['cx', 'cy']].to_numpy(dtype=float)
    return out


def prepare_roi(gt, mpp, clicks):
    """
        One ROI's ground truth and clicks, ready to simulate.

        gt (pd.DataFrame): the ROI's annotations, both categories (ann_id, cx, cy, category_id).
        mpp (float): microns per pixel.
        clicks (list[dict]): prepare_click results, in seed_index order.

        Returns dict: gt, gt_xy, gt_ann, gt_mit, ann_pos, radius, R0 and clicks.
    """
    gt = gt[['ann_id', 'cx', 'cy', 'category_id']].reset_index(drop=True)
    assert gt['ann_id'].is_unique
    return dict(gt=gt, gt_xy=gt[['cx', 'cy']].to_numpy(dtype=float), gt_ann=gt['ann_id'].to_numpy(), gt_mit=(gt['category_id'] == ds.MITOTIC).to_numpy(), ann_pos={int(a): i for i, a in enumerate(gt['ann_id'])}, radius=ev.radius_px(mpp), R0=int((gt['category_id'] == ds.MITOTIC).sum()), clicks=clicks)


def simulate_session(roi, order, arm, K, variant):
    """
        One click-review session on one ROI: each click in turn is boxed, its remaining pool ranked, the top K shown and matched, and everything shown hidden from later pages.

        roi (dict): a prepare_roi result.
        order (sequence[int]): click positions into roi['clicks'], one per iteration.
        arm (str): the ranking arm.
        K (int): the page size.
        variant (str): 'recomputed' ranks the remaining pool afresh; 'fullpool' keeps the full-pool order and skips removed items; 'refill' (unfiltered arms) takes the first n_pool survivors of the click's deep list, so hidden candidates are replaced by deeper ones.

        Returns list[dict]: one record per page, in iteration order.
    """
    radius, R0, gt, gt_xy, gt_ann, gt_mit, ann_pos = roi['radius'], roi['R0'], roi['gt'], roi['gt_xy'], roi['gt_ann'], roi['gt_mit'], roi['ann_pos']
    b_name = arm_base(arm)
    found, claimed = set(), set()
    shown = np.zeros((0, 2))
    pages = []
    for j, ci in enumerate(order):
        c = roi['clicks'][ci]
        s = c['seed_ann_id']
        assert gt_mit[ann_pos[s]], 'a click is not on a mitosis'
        r_pre = R0 - len(found)
        already = s in found
        found_before = frozenset(found)
        found.add(s)
        r = R0 - len(found)
        if r == 0:
            break
        base, xy = (c['deep'], c['deep_xy']) if variant == 'refill' else (c['base'][b_name], c['xy'][b_name])
        rm_shown = within(xy, shown, radius)
        others = [a for a in found if a != s]
        rm_found = within(xy, gt_xy[[ann_pos[a] for a in others]], radius) if others else np.zeros(len(xy), dtype=bool)
        removed = rm_shown | rm_found
        avail = ~removed
        if variant == 'refill':
            assert arm in REFILL_ARMS
            keep = np.zeros(len(base), dtype=bool)
            keep[np.flatnonzero(avail)[:c['n_pool']]] = True
            ranked = rank_remaining(arm, base[keep]) if keep.any() else np.zeros(0, dtype=int)
        else:
            keep = avail
            full = c['full_order'][arm]
            skip = full[keep[full]]
            if variant == 'recomputed':
                ranked = rank_remaining(arm, base[keep]) if keep.any() else np.zeros(0, dtype=int)
                if arm not in COMBOS:
                    assert np.array_equal(ranked, skip), f'{arm}: the recomputed order differs from the full-pool order with skips'
            else:
                ranked = skip
        excluded = found | claimed
        ev_mask = ~np.isin(gt_ann, list(excluded))
        u_xy = gt_xy[ev_mask & gt_mit]
        near_kept = within(u_xy, xy[keep], radius)
        hidden = within(u_xy, xy[removed], radius) & ~within(u_xy, xy[avail], radius)
        n = min(K, len(ranked))
        rec = dict(iteration=j + 1, seed_index=c['seed_index'], seed_ann_id=s, seed_already_found=already, r_pre_box=r_pre, r=r, n_base=len(base), n_removed_shown=int(rm_shown.sum()), n_removed_found=int(rm_found.sum()), n_removed=int(removed.sum()), n_remaining=int(keep.sum()), n=n, mitoses_reachable=int(near_kept.sum()), hidden_mitoses=int(hidden.sum()), hidden_near_shown=int((hidden & within(u_xy, shown, radius)).sum()))
        if n == 0:
            rec.update(tp=0, n_lookalike=0, n_unlabelled=0, skipped=True)
            pages.append(rec)
            continue
        page = base.iloc[ranked[:n]]
        pxy = page[['cx', 'cy']].to_numpy(dtype=float)
        assert not within(pxy, shown, radius).any(), 'a shown candidate lies within R of one shown on an earlier page'
        out, _ = ev.bucket_detections(page[['cx', 'cy']].reset_index(drop=True), gt[ev_mask].reset_index(drop=True), radius)
        bucket = out['bucket'].to_numpy()
        matched = [int(a) for a in out['matched_ann_id'].to_numpy() if a >= 0]
        assert len(set(matched)) == len(matched) and not set(matched) & excluded, 'an annotation was matched twice'
        new_mit = {a for a in matched if gt_mit[ann_pos[a]]}
        tp = int((bucket == HCL).sum())
        assert tp == len(new_mit) and tp <= r, 'TP is inconsistent with the matched mitoses or exceeds r'
        claimed |= set(matched)
        found |= new_mit
        assert found_before <= found
        shown = np.vstack([shown, pxy])
        rec.update(tp=tp, n_lookalike=int((bucket == HRL).sum()), n_unlabelled=int((bucket == NHF).sum()), skipped=False)
        pages.append(rec)
    return pages


print('simulator defined')

simulator defined


## Gate 6: the toy test, before any real data

The toy ROI has 5 mitoses (A, B, C, E and G) and 1 look-alike (D), at 0.25 µm/px, so R = 30 px. It has 3 clicks:
- **click 0 on A.** Its pool holds:
  - a candidate near A itself, which stays and is a false positive;
  - one near B;
  - one near D;
  - one far away;
  - a refused candidate near C, which only the unfiltered arms see.
- **click 1 on B.** Its pool holds:
  - a candidate within R of click 0's B candidate;
  - one near A but more than R from click 0's A candidate;
  - one near B but more than R from click 0's B candidate;
  - one near the look-alike D but more than R from click 0's D candidate;
  - one on C.
- **click 2 on C.** Its whole pool lies within R of click 0's shown candidates.

G sits within R of click 0's B candidate only. So G is reachable on page 1 but is hidden on the page that removes that
candidate.

Every expected record below was derived by hand.

In [7]:
TOY_GT = pd.DataFrame(dict(ann_id=[1, 2, 3, 4, 5, 6], cx=[100.0, 300.0, 500.0, 700.0, 1500.0, 290.0], cy=[100.0, 100.0, 100.0, 100.0, 1500.0, 75.0], category_id=[1, 1, 1, 2, 1, 1]))   # A B C D(look-alike) E G


def toy_pool(pts):
    """
        A toy click pool in tm_rank order, with constant feature columns so C1 orders it like od.

        pts (list[tuple]): (cx, cy, od, accepted) rows; score is set to od for accepted rows, and refused rows go last in tm order.

        Returns pd.DataFrame: a pool with every KEEP_COLS column.
    """
    df = pd.DataFrame(pts, columns=['cx', 'cy', 'od', 'accepted'])
    df['score'] = np.where(df['accepted'], df['od'], 0.01)
    df = df.sort_values('score', ascending=False, kind='mergesort').reset_index(drop=True)
    df['tm_rank'] = np.arange(len(df))
    for t in TERMS:
        if COLUMN[t] not in df:
            df[COLUMN[t]] = 0.5
    return df


TOY_POOLS = [toy_pool([(500, 105, 1.0, False), (110, 100, 0.9, True), (305, 100, 0.8, True), (705, 100, 0.7, True), (900, 900, 0.1, True)]), toy_pool([(308, 110, 0.99, True), (75, 100, 0.95, True), (272, 100, 0.9, True), (672, 100, 0.85, True), (505, 100, 0.5, True)]), toy_pool([(112, 102, 0.9, True), (303, 98, 0.8, True)])]
TOY_ARMS = ['od', 'C1', 'od_unfiltered']
# deep lists: click 1's continues past its pool with 4 far rows, the last of them (od 1.0) beyond the refill cap of 5 survivors; q2 sits on E
TOY_DEEP = [TOY_POOLS[0][['cx', 'cy', 'score', 'od']], pd.concat([TOY_POOLS[1][['cx', 'cy', 'score', 'od']], pd.DataFrame(dict(cx=[1000.0, 1490.0, 2000.0, 2500.0], cy=[1000.0, 1500.0, 2000.0, 2500.0], score=[0.4, 0.3, 0.2, 0.1], od=[0.97, 0.98, 0.99, 1.0]))], ignore_index=True), TOY_POOLS[2][['cx', 'cy', 'score', 'od']]]
TOY = prepare_roi(TOY_GT, 0.25, [prepare_click(p, i, a, TOY_ARMS, d.assign(rank=np.arange(len(d)))) for i, (p, a, d) in enumerate(zip(TOY_POOLS, [1, 2, 3], TOY_DEEP))])
assert TOY['radius'] == 30.0 and TOY['R0'] == 5
FIELDS = ('r', 'n', 'tp', 'n_lookalike', 'n_unlabelled', 'seed_already_found', 'n_removed_shown', 'n_removed_found', 'n_removed', 'mitoses_reachable', 'hidden_mitoses', 'hidden_near_shown', 'skipped')
TOY_EXPECTED = {
    ('od', (0, 1)): [(4, 3, 1, 1, 1, False, 0, 0, 0, 2, 0, 0, False), (3, 3, 1, 0, 2, True, 1, 1, 2, 1, 0, 0, False)],
    ('C1', (0, 1)): [(4, 3, 1, 1, 1, False, 0, 0, 0, 2, 0, 0, False), (3, 3, 1, 0, 2, True, 1, 1, 2, 1, 0, 0, False)],
    ('od', (1, 0)): [(4, 3, 1, 0, 2, False, 0, 0, 0, 2, 0, 0, False), (3, 3, 0, 1, 2, True, 1, 1, 1, 0, 1, 0, False)],
    ('od_unfiltered', (0, 1)): [(4, 3, 2, 0, 1, False, 0, 0, 0, 3, 0, 0, False), (2, 2, 0, 1, 1, True, 2, 2, 3, 0, 0, 0, False)],
    ('od', (0, 2)): [(4, 3, 1, 1, 1, False, 0, 0, 0, 2, 0, 0, False), (2, 0, 0, 0, 0, False, 2, 2, 2, 0, 1, 1, True)],
}
for (arm, order), expected in TOY_EXPECTED.items():
    for variant in variants_for(arm):
        if variant == 'refill':
            continue
        got = [tuple(p[f] for f in FIELDS) for p in simulate_session(TOY, order, arm, 3, variant)]
        assert got == expected, f'toy {arm} {order} {variant}: got {got}, expected {expected}'
# refill: page 2 (click 1) loses p1e, p1a, p1d as before, but the deep list refills to 5 survivors (p1b, p1c, q1, q2, q3; q4 is past the cap) and od ranks q3, q2, q1 first
TOY_REFILL = [(4, 3, 2, 0, 1, False, 0, 0, 0, 3, 0, 0, False), (2, 3, 1, 0, 2, True, 2, 2, 3, 1, 0, 0, False)]
got = [tuple(p[f] for f in FIELDS) for p in simulate_session(TOY, (0, 1), 'od_unfiltered', 3, 'refill')]
assert got == TOY_REFILL, f'toy refill: got {got}, expected {TOY_REFILL}'
print(f'gate 6 -- toy test: {len(TOY_EXPECTED) + 1} hand-derived sessions reproduce exactly (both variants for C1, and the refill)')

gate 6 -- toy test: 6 hand-derived sessions reproduce exactly (both variants for C1, and the refill)


## Gates 1, 2 and 4 on the real pools, and the per-ROI preparation

- **Gate 4.**
  - Every click's ROI holds `n_gt_mitotic + 1` category-1 annotations in the JSON.
  - The clicked annotation is category 1.
  - The clicks CSV's `image_id` equals the JSON's.
- **Gate 1.** Each click's pool has NOTEBOOK's `n_pool` rows and `n_accepted` accepted rows. Each ROI's 6 clicks are
  distinct annotations. A stored position is NaN exactly on refused rows.
- **Gate 2.**
  - NOTEBOOK's own `arm_lists`, run on the accepted pool, reproduces every stored position column: `pos_od`,
    `pos_tm`, `pos_C1` to `pos_C5`, both F5 directions included.
  - The simulator's full-pool orders equal the stored ones.
  - `tm_unfiltered`'s full order is the tm_rank order.
  - `od_unfiltered` has no stored positions, so gate 3 is its only check.

In [8]:
POS_COLS = {('od', 0): 'pos_od', ('tm', 0): 'pos_tm', ('C1', 0): 'pos_C1', ('C2', 0): 'pos_C2', ('C3', 1): 'pos_C3_f5up', ('C3', -1): 'pos_C3_f5down', ('C4', 0): 'pos_C4', ('C5', 1): 'pos_C5_f5up', ('C5', -1): 'pos_C5_f5down'}
IMG = images.set_index('file_name')
OD_RUN = PER_RUN[PER_RUN['arm'] == 'od'].set_index(['file_name', 'seed_index']).sort_index()
POOL_BY_CLICK = {key: g for key, g in POOL.groupby(['file_name', 'seed_index'])}
DEEP_BY_CLICK = {key: g for key, g in DEEP.groupby(['file_name', 'seed_index'])}
ROIS = {}
n_pos = 0
t0 = time.time()
for f in ROI_ORDER:
    gt = ds.image_annotations(annotations, f)
    R0 = int((gt['category_id'] == ds.MITOTIC).sum())
    meta = CLICKS[CLICKS['file_name'] == f].sort_values('seed_index')
    assert len(meta) == N_CLICKS_PER_ROI and meta['seed_ann_id'].is_unique, f'{f}: gate 1 -- the clicks are not 6 distinct annotations'
    assert meta['image_id'].nunique() == 1 and int(meta['image_id'].iat[0]) == int(IMG.loc[f, 'image_id']), f'{f}: image_id differs from the JSON'
    assert meta['mpp'].nunique() == 1
    clicks = []
    for m in meta.itertuples():
        s = int(m.seed_index)
        run = OD_RUN.loc[(f, s)]
        assert R0 == int(run['n_gt_mitotic']) + 1, f'{f} s{s}: gate 4 -- JSON category-1 count {R0} != n_gt_mitotic + 1'
        seed = gt[gt['ann_id'] == m.seed_ann_id]
        assert len(seed) == 1 and int(seed['category_id'].iat[0]) == ds.MITOTIC and int(run['seed_ann_id']) == int(m.seed_ann_id), f'{f} s{s}: the clicked annotation is not one category-1 annotation'
        frame = POOL_BY_CLICK[(f, s)]
        assert len(frame) == int(run['n_pool']) and int(frame['accepted'].sum()) == int(run['n_accepted']) and int(m.n_pool) == len(frame) and int(m.n_accepted) == int(frame['accepted'].sum()), f'{f} s{s}: gate 1 -- pool size or accepted count differs'
        assert not frame.duplicated(['cx', 'cy']).any(), f'{f} s{s}: two candidates at one location'
        pos = list(POS_COLS.values())
        assert frame.loc[~frame['accepted'], pos].isna().all().all() and frame.loc[frame['accepted'], pos].notna().all().all(), f'{f} s{s}: a position is NaN on an accepted row or set on a refused one'
        c = prepare_click(frame, s, m.seed_ann_id, SIM_ARMS, DEEP_BY_CLICK[(f, s)])
        acc = c['base']['accepted']
        acc_frame = frame.sort_values('tm_rank', kind='mergesort')
        acc_frame = acc_frame[acc_frame['accepted']].reset_index(drop=True)
        lists = arm_lists(acc)
        stored = {}
        for key, col in POS_COLS.items():
            p = acc_frame[col].to_numpy()
            assert np.array_equal(np.sort(p), np.arange(len(p))), f'{f} s{s}: {col} is not a permutation'
            stored[key] = np.argsort(p, kind='stable')
            assert np.array_equal(lists[key]['row'].to_numpy(), stored[key]), f'{f} s{s}: gate 2 -- arm_lists does not reproduce {col}'
            n_pos += 1
        for arm in SIM_ARMS:
            if arm in ('od', 'tm') or arm in COMBOS:
                assert np.array_equal(c['full_order'][arm], stored[(arm, 0)]), f'{f} s{s}: the simulator\'s full {arm} order differs from the stored positions'
        if 'tm_unfiltered' in SIM_ARMS:
            assert np.array_equal(c['full_order']['tm_unfiltered'], np.arange(len(c['base']['all']))), f'{f} s{s}: the whole pool is not in descending-score order'
        clicks.append(c)
    roi = prepare_roi(gt, float(meta['mpp'].iat[0]), clicks)
    roi.update(file_name=f, domain=DOMAIN_OF[f], image_id=int(meta['image_id'].iat[0]), bin=density_bin(R0))
    ROIS[f] = roi
ROI_INFO = pd.DataFrame([dict(file_name=f, domain=r['domain'], R0=r['R0'], bin=r['bin'], image_id=r['image_id'], radius_px=r['radius']) for f, r in ROIS.items()])
print(f'gates 1, 2 and 4 held on {len(ROIS)} ROIs and {sum(len(r["clicks"]) for r in ROIS.values())} clicks ({n_pos} stored position columns reproduced exactly) in {time.time() - t0:.1f} s')
print('refill input: every click\'s deep list (1000 rows, descending score) begins with its saved pool exactly (cx, cy, score, od)')
print('ROIs per density bin (total mitoses R0): ' + ', '.join(f'{b} {int((ROI_INFO["bin"] == b).sum())}' for b in DENSITY_LABELS) + f'; R0 range {ROI_INFO["R0"].min()}-{ROI_INFO["R0"].max()} (median {ROI_INFO["R0"].median():.0f})')
AREA = images.set_index('file_name').loc[ROI_ORDER]
area_mm2 = AREA['width'] * AREA['height'] * CLICKS.drop_duplicates('file_name').set_index('file_name').loc[ROI_ORDER, 'mpp'] ** 2 / 1e6
print(f'ROI areas {area_mm2.min():.3f}-{area_mm2.max():.3f} mm^2 ({100 * (area_mm2.max() / area_mm2.min() - 1):.1f}% spread): a fixed T per ROI is not distorted by area')

gates 1, 2 and 4 held on 49 ROIs and 294 clicks (2646 stored position columns reproduced exactly) in 9.5 s
refill input: every click's deep list (1000 rows, descending score) begins with its saved pool exactly (cx, cy, score, od)
ROIs per density bin (total mitoses R0): <=20 14, 21-40 12, 41-80 11, >80 12; R0 range 15-239 (median 36)
ROI areas 1.988-2.020 mm^2 (1.6% spread): a fixed T per ROI is not distorted by area


## Gate 3: iteration 1, run on every click as a standalone page, equals NOTEBOOK's TP@K

Each of the 294 clicks gets a one-iteration session for every simulated arm, ranking variant and K. These pages go
through the simulator's own code path. They must give NOTEBOOK's `tp_at_10/20/30` exactly. They are also the exact
page-1 values used for the page-1 tables.

In [9]:
TP_NB = PER_RUN[PER_RUN['f5_direction'] == 0].set_index(['file_name', 'seed_index', 'arm'])[KCOLS].sort_index()
g3 = []
t0 = time.time()
for f in ROI_ORDER:
    roi = ROIS[f]
    for ci, c in enumerate(roi['clicks']):
        for arm in SIM_ARMS:
            for variant in variants_for(arm):
                for k in KS:
                    pages = simulate_session(roi, [ci], arm, k, variant)
                    assert len(pages) == 1 and pages[0]['r'] == roi['R0'] - 1 and pages[0]['n'] == k and not pages[0]['seed_already_found']
                    g3.append(dict(arm=arm, variant=variant, K=k, file_name=f, seed_index=c['seed_index'], n=k, tp=pages[0]['tp'], r=pages[0]['r'], R0=roi['R0'], tp_nb=int(TP_NB.loc[(f, c['seed_index'], arm), f'tp_at_{k}'])))
STANDALONE = pd.DataFrame(g3)
bad = STANDALONE[STANDALONE['tp'] != STANDALONE['tp_nb']]
assert bad.empty, f'gate 3 -- {len(bad)} standalone pages differ from NOTEBOOK:\n{bad.head(20)}'
STANDALONE['p'] = STANDALONE['tp'] / STANDALONE['n']
chk = STANDALONE.groupby(['arm', 'variant', 'K'], sort=False)['tp'].sum().unstack('K')
print(f'gate 3 -- {len(STANDALONE)} standalone first pages ({time.time() - t0:.1f} s) reproduce NOTEBOOK\'s tp_at_10/20/30 exactly, unfiltered arms included. Pooled P@K:')
print((chk / pd.Series({k: k * N_CLICKS for k in KS})).to_string(float_format=lambda v: f'{v:.4f}'))
P_FRESH = STANDALONE.set_index(['arm', 'variant', 'K', 'file_name', 'seed_index'])['p']

gate 3 -- 6174 standalone first pages (29.0 s) reproduce NOTEBOOK's tp_at_10/20/30 exactly, unfiltered arms included. Pooled P@K:
K                            10     20     30
arm           variant                        
C1            recomputed 0.6374 0.5495 0.4822
              fullpool   0.6374 0.5495 0.4822
od_unfiltered recomputed 0.5626 0.5090 0.4430
              refill     0.5626 0.5090 0.4430
tm_unfiltered recomputed 0.5418 0.4685 0.4162
              refill     0.5418 0.4685 0.4162
od            recomputed 0.5888 0.5194 0.4523


## The cost model, re-derived

**Regime A: the tool replaces hand search.**
- Hand search makes one systematic pass of length T over a ROI holding r unfound mitoses, spread evenly. Scanning a
  fraction f of the ROI takes f·T and finds f·r of them in expectation. The manual yield is therefore r/T mitoses per
  second, and one mitosis costs T/r seconds.
- A page costs `n·v + c + L` seconds and yields TP mitoses.
- The page beats continuing by hand when `TP/(n·v + c + L) > r/T`. Setting p = TP/n, that is
  `p > p* = r·(n·v + c + L)/(n·T)`.
- `E = p/p* = TP·T/(r·(n·v + c + L))`.
- The page saves `TP·T/r − cost = (E − 1)·cost` seconds. The identity is exact.
- **Units.** p\* is (mitoses)·(s)/((candidates)·(s)) = mitoses per candidate, as a precision should be.
- **p\* ∝ r.** It is, as long as T is fixed. Hand search is fastest in dense ROIs, so a page must be more precise
  there.
- **Ceiling.** A page can be at most `p_max = min(1, r/n)` precise, so `E_max = min(n, r)·T/(r·cost)`. When r ≤ n,
  `E_max = T/cost`, which is 22 at K = 10, central. So unreachability comes only from large r.

**What the derivation exposes, logged in the verdict:**
1. **The per-page E is a marginal test.** It compares the page with hand search *now*, at the depleted rate r_j/T.
   That is the right test for "is the next page worth reviewing". Summing `(E − 1)·cost` over pages is *not* the
   time saved against a pathologist without the tool, because every later page is valued at T/r_j > T/R0.
   - Example: 10 mitoses, two pages that find 5 and then 5. The page sum values them at
     5·T/10 + 5·T/5 = 1.5·T. One hand pass finds all 10 in T.
   - The session figure `ΣTP·T/R0 − Σcost`, with r fixed at the ROI total, is the coherent "against no tool"
     number. It treats seeds as found at the no-tool rate, which flatters the tool.
   - Subtracting the extra cost of hand-finding later seeds in a depleted ROI, `T/r_pre − T/R0` per new seed, gives
     the third accounting.
2. **The box.** Its cost c is charged in full to the page, while finding the seed is treated as neutral. A manual
   pass would also spend time marking that mitosis, so charging all of c is slightly conservative. Nothing is double
   counted.
3. **Rate against order statistic.** The expected time to find m of r evenly spread mitoses by a systematic scan is
   m·T/(r+1), not m·T/r. The difference is a factor (r+1)/r: 2 at r = 1, 1.1 at r = 10.
4. **T fixed per ROI.** ROI areas differ by under 2 %, so area is not a confound. Density is: a dense ROI takes
   longer to pass because every mitosis must be decided and marked.
   - The sensitivity `T_eff = T + r·c_saved` bounds `p*_A` by `cost/(n·c_saved)`, which is regime B's `p*_B`. Dense
     ROIs then become reachable.
5. **Regime A presumes a partial-yield goal.** If the goal is an exhaustive count, a full pass is mandatory. Then
   regime B is the right model, and regime A's savings do not exist.

**Regime B: a full manual pass is mandatory anyway.** A tool-found mitosis saves only its decide-and-mark time
`c_saved`, so `p*_B = (n·v + c + L)/(n·c_saved)`. It does not depend on r.

In [10]:
def regime_a(n, tp, r, R0, v, T, c=C_BOX, L=L_WAIT):
    """
        Regime A per page, the tool replacing hand search: break-even precision, efficiency and its ceiling, reachability, and seconds saved under two accountings.

        n (array-like): candidates shown; 0 for a skipped page.
        tp (array-like): mitoses confirmed on the page.
        r (array-like): mitoses still unfound when the page is shown (> 0).
        R0 (array-like): the ROI's total mitoses, for the session accounting.
        v (float): seconds to judge one candidate.
        T (float): seconds for one complete manual pass.
        c (float): seconds to draw the box.
        L (float): seconds of latency.

        Returns dict: cost, p_star, E, E_max, reach, saved_page (the prompt's, at T/r) and saved_session (at T/R0), as arrays.
    """
    n, tp, r, R0 = (np.asarray(x, dtype=float) for x in (n, tp, r, R0))
    shown = n > 0
    cost = n * v + c + L
    with np.errstate(divide='ignore', invalid='ignore'):
        p_star = np.where(shown, r * cost / (n * T), np.nan)
        E = np.where(shown, tp * T / (r * cost), 0.0)
        E_max = np.where(shown, np.minimum(n, r) * T / (r * cost), 0.0)
    reach = np.where(~shown, 'skipped', np.where(E_max < 1, 'unreachable_E1', np.where(E_max < 2, 'unreachable_E2', 'reachable')))
    return dict(cost=cost, p_star=p_star, E=E, E_max=E_max, reach=reach, saved_page=tp * T / r - cost, saved_session=tp * T / R0 - cost)


def regime_a_density(n, tp, r, v, T, c_saved, c=C_BOX, L=L_WAIT):
    """
        Regime A with a pass time that grows with density, T_eff = T + r * c_saved: efficiency and its ceiling.

        n (array-like): candidates shown; 0 for a skipped page.
        tp (array-like): mitoses confirmed.
        r (array-like): mitoses still unfound.
        v (float): seconds to judge one candidate.
        T (float): seconds for the density-free part of a manual pass.
        c_saved (float): seconds to decide and mark one mitosis inside the pass.
        c (float): seconds to draw the box.
        L (float): seconds of latency.

        Returns dict: p_star, E and E_max, as arrays.
    """
    n, tp, r = (np.asarray(x, dtype=float) for x in (n, tp, r))
    shown = n > 0
    cost = n * v + c + L
    T_eff = T + r * c_saved
    with np.errstate(divide='ignore', invalid='ignore'):
        return dict(p_star=np.where(shown, r * cost / (n * T_eff), np.nan), E=np.where(shown, tp * T_eff / (r * cost), 0.0), E_max=np.where(shown, np.minimum(n, r) * T_eff / (r * cost), 0.0))


def regime_b(n, tp, r, v, c_saved, c=C_BOX, L=L_WAIT):
    """
        Regime B per page, a full manual pass being mandatory: break-even precision, efficiency, reachability and seconds saved.

        n (array-like): candidates shown; 0 for a skipped page.
        tp (array-like): mitoses confirmed.
        r (array-like): mitoses still unfound (only for the precision ceiling).
        v (float): seconds to judge one candidate.
        c_saved (float): seconds a tool-found mitosis saves inside the pass.
        c (float): seconds to draw the box.
        L (float): seconds of latency.

        Returns dict: p_star, E, reachable (p_max >= p_star) and saved, as arrays.
    """
    n, tp, r = (np.asarray(x, dtype=float) for x in (n, tp, r))
    shown = n > 0
    cost = n * v + c + L
    with np.errstate(divide='ignore', invalid='ignore'):
        p_star = np.where(shown, cost / (n * c_saved), np.nan)
        p_max = np.where(shown, np.minimum(1.0, r / n), 0.0)
    return dict(p_star=p_star, E=np.where(shown, tp * c_saved / cost, 0.0), reachable=shown & (p_max >= p_star), saved=tp * c_saved - cost)


T0 = 60.0 * T0_MIN
# the identities, on random pages
rng_chk = np.random.default_rng(0)
n_, r_ = rng_chk.integers(1, 31, 1000), rng_chk.integers(1, 250, 1000)
tp_ = np.minimum(rng_chk.integers(0, 31, 1000), np.minimum(n_, r_))
for v_ in V_GRID:
    for tm_ in T_GRID_MIN:
        A = regime_a(n_, tp_, r_, r_ + 5, v_, 60.0 * tm_)
        assert np.allclose(A['E'] * A['p_star'], tp_ / n_) and np.allclose(A['saved_page'], (A['E'] - 1) * A['cost']) and np.allclose(regime_a(n_, tp_, 2 * r_, r_, v_, 60.0 * tm_)['p_star'], 2 * A['p_star'])
        assert np.allclose(A['E_max'], np.minimum(1, r_ / n_) / A['p_star'])
        D = regime_a_density(n_, tp_, r_, v_, 60.0 * tm_, C_SAVED0)
        assert (D['p_star'] < (n_ * v_ + C_BOX + L_WAIT) / (n_ * C_SAVED0)).all(), 'the density-T p* must stay below regime B p*'
print('identities hold on 1000 random pages x 9 settings: E * p* = p; seconds saved = (E - 1) * cost; p* doubles when r doubles; E_max = p_max / p*; density-T p* < p*_B')

print('\nr above which a full page (n = K) cannot reach the target at the central setting (v 2 s, T 10 min, c 5 s, L 2 s):')
for k in KS:
    cost = k * V0 + C_BOX + L_WAIT
    print(f'  K={k}: cost {cost:.0f} s; p* = {cost / (k * T0):.5f} * r; E>=2 unreachable for r > {k * T0 / (2 * cost):.1f}, E>=1 unreachable for r > {k * T0 / cost:.1f}')
print('  (the prompt quotes "about 111" for E>=2 at K=10: confirmed)')

# the summed-page accounting overstates against a no-tool baseline: 10 mitoses, two full pages finding 5 and then 5
pa = regime_a([10, 10], [5, 5], [10, 5], [10, 10], V0, T0)
print(f"\ntwo-page example, R0 = r = 10, pages find 5 then 5 (central): value by the page sum {(np.array([5, 5]) * T0 / np.array([10, 5])).sum():.0f} s = {(np.array([5, 5]) * T0 / np.array([10, 5])).sum() / T0:.2f} T; one manual pass finds all 10 in T = {T0:.0f} s; session value {10 * T0 / 10:.0f} s. Seconds saved: page sum {pa['saved_page'].sum():.0f}, session {pa['saved_session'].sum():.0f}")
print('order statistic vs rate model, expected time to find 1 of r: ' + ', '.join(f'r={r}: T/(r+1) / (T/r) = {r / (r + 1):.3f}' for r in (1, 2, 5, 10, 40)))

# the targets table: p* and 2 p* against r, for full pages, at every setting; regime B and the density-T variant beside it
trows = []
for k in KS:
    for v in V_GRID:
        for tmin in T_GRID_MIN:
            for r in R_TABLE:
                A = regime_a(k, 0, r, r, v, 60.0 * tmin)
                p_max = min(1.0, r / k)
                trows.append(dict(regime='A', K=k, r=r, v=v, T_min=tmin, c_saved=np.nan, cost_s=float(A['cost']), p_star=float(A['p_star']), two_p_star=2 * float(A['p_star']), p_max=p_max, unreachable_E1=float(A['p_star']) > p_max, unreachable_E2=2 * float(A['p_star']) > p_max, central=(v, tmin) == (V0, T0_MIN)))
                for cs in C_SAVED_GRID:
                    D = regime_a_density(k, 0, r, v, 60.0 * tmin, cs)
                    trows.append(dict(regime='A_density_T', K=k, r=r, v=v, T_min=tmin, c_saved=cs, cost_s=float(A['cost']), p_star=float(D['p_star']), two_p_star=2 * float(D['p_star']), p_max=p_max, unreachable_E1=float(D['p_star']) > p_max, unreachable_E2=2 * float(D['p_star']) > p_max, central=(v, tmin, cs) == (V0, T0_MIN, C_SAVED0)))
        for cs in C_SAVED_GRID:
            B = regime_b(k, 0, k, v, cs)
            trows.append(dict(regime='B', K=k, r=np.nan, v=v, T_min=np.nan, c_saved=cs, cost_s=k * v + C_BOX + L_WAIT, p_star=float(B['p_star']), two_p_star=2 * float(B['p_star']), p_max=1.0, unreachable_E1=float(B['p_star']) > 1, unreachable_E2=2 * float(B['p_star']) > 1, central=(v, cs) == (V0, C_SAVED0)))
TARGETS = pd.DataFrame(trows)
assert (TARGETS.loc[TARGETS['regime'] == 'A', 'unreachable_E2'] == (TARGETS.loc[TARGETS['regime'] == 'A', 'two_p_star'] > 1)).all(), 'in this grid, p* > p_max only when p* > 1 (cost < T/2 always)'


def target_cell(p):
    """
        One printable target cell, flagged when no page can reach it.

        p (float): a target precision.

        Returns str: the value, or the value marked UNREACHABLE when it exceeds 1.
    """
    return f'{p:.3f}' if p <= 1 else f'{p:.2f} UNREACH'


cen = TARGETS[(TARGETS['regime'] == 'A') & TARGETS['central']]
tab = pd.DataFrame({f'K={k} {lab}': cen[cen['K'] == k].set_index('r')[col].map(target_cell) for k in KS for lab, col in (('p*', 'p_star'), ('2p*', 'two_p_star'))})
print('\nTARGETS, regime A, central setting (v 2 s, T 10 min): break-even p* and target 2 p* by unfound mitoses r, full pages:')
print(tab.to_string())
gA = TARGETS[TARGETS['regime'] == 'A'].groupby(['r', 'K'])['two_p_star'].agg(['min', 'max']).unstack('K')
print('\n2 p* across the 9-setting grid (v 1.5-3 s x T 5-20 min), min-max; values above 1 are unreachable:')
print(pd.DataFrame({f'K={k}': [f"{gA.loc[r, ('min', k)]:.3f}-{gA.loc[r, ('max', k)]:.3f}" + (' (UNREACH at worst)' if gA.loc[r, ('max', k)] > 1 else '') for r in R_TABLE] for k in KS}, index=pd.Index(R_TABLE, name='r')).to_string())
cd = TARGETS[(TARGETS['regime'] == 'A_density_T') & TARGETS['central']]
print('\nsensitivity, T grows with density (T_eff = 10 min + r x 6 s), 2 p*: ' + '; '.join(f'K={k}: ' + ', '.join(f'r={int(x.r)} {x.two_p_star:.3f}' for x in cd[cd['K'] == k].itertuples()) for k in KS))
cb = TARGETS[(TARGETS['regime'] == 'B') & (TARGETS['v'] == V0)]
print('regime B (full pass mandatory), p*_B at v 2 s: ' + '; '.join(f'K={k}: ' + ', '.join(f'c_saved {x.c_saved:.0f}s {x.p_star:.3f}' for x in cb[cb['K'] == k].itertuples()) for k in KS))

identities hold on 1000 random pages x 9 settings: E * p* = p; seconds saved = (E - 1) * cost; p* doubles when r doubles; E_max = p_max / p*; density-T p* < p*_B

r above which a full page (n = K) cannot reach the target at the central setting (v 2 s, T 10 min, c 5 s, L 2 s):
  K=10: cost 27 s; p* = 0.00450 * r; E>=2 unreachable for r > 111.1, E>=1 unreachable for r > 222.2
  K=20: cost 47 s; p* = 0.00392 * r; E>=2 unreachable for r > 127.7, E>=1 unreachable for r > 255.3
  K=30: cost 67 s; p* = 0.00372 * r; E>=2 unreachable for r > 134.3, E>=1 unreachable for r > 268.7
  (the prompt quotes "about 111" for E>=2 at K=10: confirmed)

two-page example, R0 = r = 10, pages find 5 then 5 (central): value by the page sum 900 s = 1.50 T; one manual pass finds all 10 in T = 600 s; session value 600 s. Seconds saved: page sum 846, session 546
order statistic vs rate model, expected time to find 1 of r: r=1: T/(r+1) / (T/r) = 0.500, r=2: T/(r+1) / (T/r) = 0.667, r=5: T/(r+1) / (T/r) = 0.833, r=10

## The run

The run covers 49 ROIs × 20 orders × K = 10, 20 and 30 × 6 iterations, with these arms and variants:
- `C1`, recomputed and fullpool;
- `od_unfiltered`;
- `tm_unfiltered`;
- `od`.

The invariants are asserted on every page.

In [11]:
ORDERS = {}
for f in ROI_ORDER:
    rng = np.random.default_rng([ORDER_SEED, ROIS[f]['image_id']])
    ORDERS[f] = [rng.permutation(N_CLICKS_PER_ROI) for _ in range(N_ORDERS)]
first_counts = pd.Series([int(o[0]) for f in ROI_ORDER for o in ORDERS[f]]).value_counts().sort_index()
print(f'orders: {N_ORDERS} per ROI; first-click position counts over all ROIs {first_counts.to_dict()} (uniform would be {N_ORDERS * len(ROI_ORDER) / N_CLICKS_PER_ROI:.0f} each)')

rows = []
t0 = time.time()
for i, f in enumerate(ROI_ORDER):
    roi = ROIS[f]
    for arm in SIM_ARMS:
        for variant in variants_for(arm):
            for k in KS:
                for o, order in enumerate(ORDERS[f]):
                    for rec in simulate_session(roi, order, arm, k, variant):
                        rows.append(dict(arm=arm, variant=variant, K=k, file_name=f, order=o, **rec))
    if (i + 1) % 7 == 0:
        print(f'[{i + 1}/{len(ROI_ORDER)}] {(time.time() - t0) / 60:.1f} min, {len(rows)} pages', flush=True)
PAGES = pd.DataFrame(rows).merge(ROI_INFO[['file_name', 'domain', 'R0', 'bin']], on='file_name', how='left', validate='many_to_one')
print(f'{len(PAGES)} page records in {(time.time() - t0) / 60:.1f} min; every invariant held on every page')

orders: 20 per ROI; first-click position counts over all ROIs {0: 170, 1: 156, 2: 174, 3: 149, 4: 164, 5: 167} (uniform would be 163 each)


[7/49] 1.6 min, 17505 pages


[14/49] 3.1 min, 35145 pages


[21/49] 4.9 min, 52785 pages


[28/49] 6.4 min, 69325 pages


[35/49] 8.2 min, 86965 pages


[42/49] 10.0 min, 104605 pages


[49/49] 11.7 min, 122245 pages


122245 page records in 11.7 min; every invariant held on every page


In [12]:
# per-page metrics at the central setting
PAGES['shown'] = PAGES['n'] > 0
PAGES['p'] = np.where(PAGES['shown'], PAGES['tp'] / PAGES['n'].where(PAGES['shown'], 1), np.nan)
PAGES['p_fresh'] = [P_FRESH.get((a, v, k, f, s), np.nan) for a, v, k, f, s in zip(PAGES['arm'], PAGES['variant'], PAGES['K'], PAGES['file_name'], PAGES['seed_index'])]
A0 = regime_a(PAGES['n'], PAGES['tp'], PAGES['r'], PAGES['R0'], V0, T0)
PAGES['cost_s'] = A0['cost']
PAGES['p_star'] = A0['p_star']
PAGES['two_p_star'] = 2 * A0['p_star']
PAGES['E'] = A0['E']
PAGES['E_max'] = A0['E_max']
PAGES['reach'] = A0['reach']
PAGES['saved_page_s'] = A0['saved_page']
PAGES['saved_session_s'] = A0['saved_session']
PAGES['seed_premium_s'] = np.where(PAGES['seed_already_found'], 0.0, T0 / PAGES['r_pre_box'] - T0 / PAGES['R0'])
PAGES['saved_session_seed_s'] = PAGES['saved_session_s'] - PAGES['seed_premium_s']
B0 = regime_b(PAGES['n'], PAGES['tp'], PAGES['r'], V0, C_SAVED0)
PAGES['p_star_B'] = B0['p_star']
PAGES['E_B'] = B0['E']
PAGES['reach_B'] = B0['reachable']
PAGES['saved_B_s'] = B0['saved']
D0 = regime_a_density(PAGES['n'], PAGES['tp'], PAGES['r'], V0, T0, C_SAVED0)
PAGES['E_dens'] = D0['E']
PAGES['E_max_dens'] = D0['E_max']
shown_rows = PAGES['shown'].to_numpy()
assert np.isfinite(PAGES.loc[shown_rows, ['p_star', 'E', 'E_max', 'p_star_B', 'E_B']].to_numpy()).all(), 'gate 5 -- p* or E is not finite on a shown page'
assert (PAGES['tp'] <= PAGES['r']).all() and (PAGES['n'] <= PAGES['K']).all()
assert (PAGES.loc[PAGES['iteration'] == 1, 'seed_premium_s'] == 0).all()

# sanity: the simulated iteration 1 is the click's standalone page
it1 = PAGES[PAGES['iteration'] == 1]
assert (it1['p'] == it1['p_fresh']).all() and (it1['r'] == it1['R0'] - 1).all(), 'iteration 1 is not the standalone page of its click'

PAGES['tp0'] = np.where(PAGES['shown'], (PAGES['tp'] == 0).astype(float), np.nan)
PAGES['under'] = (PAGES['n'] < PAGES['K']).astype(float)
PAGES['e1'] = (PAGES['E'] >= 1).astype(float)
PAGES['e2'] = (PAGES['E'] >= 2).astype(float)
PAGES['e1_reach'] = np.where(PAGES['E_max'] >= 1, PAGES['e1'], np.nan)
PAGES['e2_reach'] = np.where(PAGES['E_max'] >= 2, PAGES['e2'], np.nan)
PAGES['eB1'] = (PAGES['E_B'] >= 1).astype(float)
PAGES['eB1_reach'] = np.where(PAGES['reach_B'], PAGES['eB1'], np.nan)
PAGES['ed1'] = (PAGES['E_dens'] >= 1).astype(float)
PAGES['ed2'] = (PAGES['E_dens'] >= 2).astype(float)
PAGES['ed2_reach'] = np.where(PAGES['E_max_dens'] >= 2, PAGES['ed2'], np.nan)
PAGES['coverage'] = np.where(PAGES['r'] > 0, PAGES['mitoses_reachable'] / PAGES['r'], np.nan)
cap = np.minimum(PAGES['n'], PAGES['mitoses_reachable'])
PAGES['ceiling'] = np.where(PAGES['shown'], cap / PAGES['n'].where(PAGES['shown'], 1), np.nan)
PAGES['efficiency'] = np.where(PAGES['shown'] & (cap > 0), PAGES['tp'] / cap.where(cap > 0, 1), np.nan)
PAGES['seed_found_it2plus'] = np.where(PAGES['iteration'] >= 2, PAGES['seed_already_found'].astype(float), np.nan)
PRIM = PAGES[PAGES['variant'] == 'recomputed']
print(f"pages: {len(PAGES)}; shown {int(PAGES['shown'].sum())}, skipped (empty pool, r > 0) {int((~PAGES['shown']).sum())}")

pages: 122245; shown 122243, skipped (empty pool, r > 0) 2


## Per-page precision by iteration

The table gives the mean over ROIs of each ROI's mean page precision, with ROI-cluster 95 % CIs. Shown pages only.
The share of zero-TP pages is beside it.

The sanity anchors:
- the simulated iteration 1 against NOTEBOOK's P@K, which is exact on the standalone pages and approximate under the
  order weighting;
- later pages falling in sparse ROIs.

In [13]:
SH = PRIM[PRIM['shown']]
prec = summarise(SH, 'p', ['arm', 'K', 'iteration'])
zero = summarise(SH, 'tp0', ['arm', 'K', 'iteration'])
collect('precision_by_iteration', 'page precision', prec)
collect('precision_by_iteration', 'share of zero-TP pages', zero)
collect('precision_by_iteration_bin', 'page precision', summarise(SH, 'p', ['arm', 'K', 'iteration', 'bin']))
collect('precision_by_iteration_bin', 'share of zero-TP pages', summarise(SH, 'tp0', ['arm', 'K', 'iteration', 'bin']))
collect('precision_by_iteration_domain', 'page precision', summarise(SH, 'p', ['arm', 'K', 'iteration', 'domain']))
collect('precision_by_iteration_domain', 'share of zero-TP pages', summarise(SH, 'tp0', ['arm', 'K', 'iteration', 'domain']))
for k in KS:
    t = prec[prec['K'] == k].assign(cell=lambda d: [fmt_ci(m, lo, hi) + f' n{n}' for m, lo, hi, n in zip(d['mean'], d['ci_lo'], d['ci_hi'], d['n_roi'])]).pivot(index='iteration', columns='arm', values='cell')[SIM_ARMS]
    z = zero[zero['K'] == k].pivot(index='iteration', columns='arm', values='mean')[SIM_ARMS]
    print(f'\nK={k}: page precision, mean over ROIs [95% CI] n_roi')
    print(t.to_string())
    print(f'K={k}: share of zero-TP pages (mean over ROIs)')
    print(z.to_string(float_format=lambda v: f'{v:.3f}'))

print('\nsanity anchor 1 -- iteration 1:')
for arm in SIM_ARMS:
    for k in KS:
        exact = STANDALONE[(STANDALONE['arm'] == arm) & (STANDALONE['variant'] == 'recomputed') & (STANDALONE['K'] == k)]['tp'].sum() / (k * N_CLICKS)
        sim = prec[(prec['arm'] == arm) & (prec['K'] == k) & (prec['iteration'] == 1)]['mean'].iat[0]
        print(f'  {arm:14s} K={k}: NOTEBOOK P@{k} {exact:.4f} (standalone pages, exact) vs simulated iteration-1 ROI mean {sim:.4f} (order-weighted), gap {sim - exact:+.4f}')

pb = summarise(SH, 'p', ['arm', 'K', 'iteration', 'bin'])
print('\nsanity anchor 2 -- page precision by density bin, C1 (recomputed), iterations 1 and 6:')
q = pb[(pb['arm'] == BEST) & pb['iteration'].isin([1, 6])].pivot_table(index=['K', 'bin'], columns='iteration', values='mean').reindex(pd.MultiIndex.from_product([KS, DENSITY_LABELS]))
q['change'] = q[6] - q[1]
print(q.to_string(float_format=lambda v: f'{v:+.3f}'))


K=10: page precision, mean over ROIs [95% CI] n_roi
arm                              C1             od_unfiltered             tm_unfiltered                        od
iteration                                                                                                        
1          0.642 [0.571, 0.709] n49  0.565 [0.491, 0.636] n49  0.543 [0.468, 0.616] n49  0.591 [0.514, 0.665] n49
2          0.477 [0.394, 0.561] n49  0.447 [0.368, 0.526] n49  0.438 [0.358, 0.520] n49  0.449 [0.366, 0.531] n49
3          0.381 [0.300, 0.464] n49  0.322 [0.248, 0.400] n49  0.349 [0.273, 0.430] n49  0.321 [0.248, 0.399] n49
4          0.307 [0.228, 0.389] n49  0.297 [0.223, 0.376] n49  0.285 [0.212, 0.363] n49  0.298 [0.223, 0.378] n49
5          0.260 [0.186, 0.339] n49  0.261 [0.185, 0.341] n49  0.240 [0.171, 0.314] n49  0.255 [0.181, 0.335] n49
6          0.216 [0.149, 0.288] n49  0.216 [0.149, 0.289] n48  0.214 [0.150, 0.285] n48  0.210 [0.145, 0.282] n48
K=10: share of zero-TP pages (mean 


sanity anchor 2 -- page precision by density bin, C1 (recomputed), iterations 1 and 6:
iteration      1      6  change
10 <=20   +0.488 +0.035  -0.453
   21-40  +0.569 +0.053  -0.516
   41-80  +0.699 +0.211  -0.488
   >80    +0.842 +0.594  -0.248
20 <=20   +0.346 +0.012  -0.335
   21-40  +0.434 +0.023  -0.411
   41-80  +0.641 +0.089  -0.552
   >80    +0.822 +0.325  -0.497
30 <=20   +0.268 +0.011  -0.258
   21-40  +0.342 +0.015  -0.327
   41-80  +0.561 +0.071  -0.489
   >80    +0.801 +0.250  -0.550


## Reaching the targets, at the central setting

The table gives the share of pages with E ≥ 1 (break-even) and with E ≥ 2 (the target), by iteration and density bin.
Each share is reported twice:
- over all pages, a skipped page counting as E = 0;
- over the pages where that target was reachable.

Judge the method on the reachable-only share. Page 1 per ROI comes from the exact standalone pages: the ROI's mean
P@K over its 6 clicks, against p\* and 2p\* at r = R0 − 1.

In [14]:
REACH = {}
for col, lab in (('e1', 'E>=1, all pages'), ('e1_reach', 'E>=1, reachable pages'), ('e2', 'E>=2, all pages'), ('e2_reach', 'E>=2, reachable pages')):
    REACH[col] = summarise(PRIM, col, ['arm', 'K', 'iteration'])
    collect('reach_central', lab, REACH[col])
    collect('reach_central_bin', lab, summarise(PRIM, col, ['arm', 'K', 'iteration', 'bin']))
    collect('reach_central_allpages', lab, summarise(PRIM, col, ['arm', 'K']))
    collect('reach_central_allpages_bin', lab, summarise(PRIM, col, ['arm', 'K', 'bin']))
cls = PRIM.assign(**{c: (PRIM['reach'] == c).astype(float) for c in ('unreachable_E1', 'unreachable_E2', 'reachable', 'skipped')})
for c in ('unreachable_E1', 'unreachable_E2', 'reachable', 'skipped'):
    collect('reach_class', f'share of pages {c}', summarise(cls, c, ['arm', 'K', 'iteration']))

for k in KS:
    print(f'\nK={k}, central: share of pages reaching E>=1 and E>=2 (mean over ROIs); "all" counts every page, "reach" only pages where the target was reachable')
    t = pd.concat({lab: REACH[col][REACH[col]['K'] == k].pivot(index='iteration', columns='arm', values='mean')[SIM_ARMS] for col, lab in (('e1', 'E>=1 all'), ('e1_reach', 'E>=1 reach'), ('e2', 'E>=2 all'), ('e2_reach', 'E>=2 reach'))}, axis=1)
    print(t.to_string(float_format=lambda v: f'{v:.3f}'))

print('\nC1 (in-sample), central: share of pages with E>=2 by density bin and iteration -- all pages / reachable pages only:')
for k in KS:
    a = summarise(PRIM[(PRIM['arm'] == BEST) & (PRIM['K'] == k)], 'e2', ['iteration', 'bin']).pivot(index='bin', columns='iteration', values='mean').reindex(DENSITY_LABELS)
    b = summarise(PRIM[(PRIM['arm'] == BEST) & (PRIM['K'] == k)], 'e2_reach', ['iteration', 'bin']).pivot(index='bin', columns='iteration', values='mean').reindex(DENSITY_LABELS)
    print(f'K={k}:')
    print(pd.DataFrame({it: [f'{a.loc[bn, it]:.2f} / ' + ('n/a' if np.isnan(b.loc[bn, it]) else f'{b.loc[bn, it]:.2f}') for bn in DENSITY_LABELS] for it in a.columns}, index=DENSITY_LABELS).to_string())
print('\nC1, central: share of pages with E>=1 by density bin and iteration -- all pages / reachable pages only:')
for k in KS:
    a = summarise(PRIM[(PRIM['arm'] == BEST) & (PRIM['K'] == k)], 'e1', ['iteration', 'bin']).pivot(index='bin', columns='iteration', values='mean').reindex(DENSITY_LABELS)
    b = summarise(PRIM[(PRIM['arm'] == BEST) & (PRIM['K'] == k)], 'e1_reach', ['iteration', 'bin']).pivot(index='bin', columns='iteration', values='mean').reindex(DENSITY_LABELS)
    print(f'K={k}:')
    print(pd.DataFrame({it: [f'{a.loc[bn, it]:.2f} / ' + ('n/a' if np.isnan(b.loc[bn, it]) else f'{b.loc[bn, it]:.2f}') for bn in DENSITY_LABELS] for it in a.columns}, index=DENSITY_LABELS).to_string())

rc = summarise(cls[cls['arm'] == BEST], 'unreachable_E2', ['K', 'iteration']).pivot(index='iteration', columns='K', values='mean')
rc1 = summarise(cls[cls['arm'] == BEST], 'unreachable_E1', ['K', 'iteration']).pivot(index='iteration', columns='K', values='mean')
print('\nC1: share of pages where E>=2 is unreachable but E>=1 is not (dense ROIs), and where even E>=1 is unreachable, by iteration:')
print(pd.concat({'unreach E2 only': rc, 'unreach E1': rc1}, axis=1).to_string(float_format=lambda v: f'{v:.3f}'))


def page1_table(arm, v, T, variant='recomputed'):
    """
        Page 1 per ROI, exactly: the mean P@K over the ROI's 6 standalone first pages, against p* and 2 p* at r = R0 - 1.

        arm (str): the arm.
        v (float): seconds per candidate.
        T (float): seconds per manual pass.
        variant (str): the ranking variant.

        Returns pd.DataFrame: one row per (K, ROI) with p, p_star, two_p_star, E_max, reach, below_break_even, below_target, clicks_below_break_even, clicks_below_target.
    """
    s = STANDALONE[(STANDALONE['arm'] == arm) & (STANDALONE['variant'] == variant)]
    A = regime_a(s['n'], s['tp'], s['r'], s['R0'], v, T)
    s = s.assign(E=A['E'])
    g = s.groupby(['K', 'file_name'], sort=False).agg(p=('p', 'mean'), r=('r', 'first'), R0=('R0', 'first'), clicks_below_break_even=('E', lambda e: int((e < 1).sum())), clicks_below_target=('E', lambda e: int((e < 2).sum()))).reset_index()
    A = regime_a(g['K'], 0, g['r'], g['R0'], v, T)
    g['p_star'], g['two_p_star'], g['E_max'], g['reach'] = A['p_star'], 2 * A['p_star'], A['E_max'], A['reach']
    g['below_break_even'] = g['p'] < g['p_star']
    g['below_target'] = g['p'] < g['two_p_star']
    return g.merge(ROI_INFO[['file_name', 'domain', 'bin']], on='file_name')


P1 = {arm: page1_table(arm, V0, T0) for arm in SIM_ARMS}
print('\npage 1 per ROI, central setting -- ROIs whose mean P@K over 6 clicks is below break-even (p < p*) and below target (p < 2 p*):')
for arm in SIM_ARMS:
    for k in KS:
        g = P1[arm][P1[arm]['K'] == k]
        reach2 = g['reach'] == 'reachable'
        print(f"  {arm:14s} K={k}: below break-even {int(g['below_break_even'].sum())}/49, below target {int(g['below_target'].sum())}/49 (of which E>=2 unreachable: {int((g['below_target'] & ~reach2).sum())}); below target among reachable ROIs {int((g['below_target'] & reach2).sum())}/{int(reach2.sum())}")
for k in KS:
    g = P1[BEST][P1[BEST]['K'] == k].sort_values('R0')
    print(f'\nC1 (in-sample) page 1 at K={k}, central: ROIs below target or with any click below break-even:')
    show = g[g['below_target'] | (g['clicks_below_break_even'] > 0)]
    print(show[['file_name', 'domain', 'R0', 'p', 'p_star', 'two_p_star', 'E_max', 'reach', 'below_break_even', 'below_target', 'clicks_below_break_even', 'clicks_below_target']].to_string(index=False, float_format=lambda v: f'{v:.3f}') if len(show) else '  none')


K=10, central: share of pages reaching E>=1 and E>=2 (mean over ROIs); "all" counts every page, "reach" only pages where the target was reachable
          E>=1 all                                   E>=1 reach                                   E>=2 all                                   E>=2 reach                                  
arm             C1 od_unfiltered tm_unfiltered    od         C1 od_unfiltered tm_unfiltered    od       C1 od_unfiltered tm_unfiltered    od         C1 od_unfiltered tm_unfiltered    od
iteration                                                                                                                                                                                
1            0.902         0.864         0.842 0.884      0.921         0.882         0.859 0.902    0.730         0.608         0.567 0.647      0.812         0.677         0.632 0.720
2            0.882         0.832         0.807 0.818      0.900         0.849         0.824 0.835    0.602   

K=30:
                 1            2            3            4            5            6
<=20   0.90 / 0.90  0.45 / 0.45  0.30 / 0.30  0.18 / 0.18  0.22 / 0.22  0.14 / 0.14
21-40  0.83 / 0.83  0.34 / 0.34  0.12 / 0.12  0.11 / 0.11  0.07 / 0.07  0.05 / 0.05
41-80  0.81 / 0.81  0.45 / 0.45  0.29 / 0.29  0.13 / 0.13  0.14 / 0.14  0.14 / 0.14
>80    0.36 / 0.48  0.48 / 0.64  0.26 / 0.32  0.16 / 0.20  0.18 / 0.20  0.15 / 0.17

C1, central: share of pages with E>=1 by density bin and iteration -- all pages / reachable pages only:
K=10:
                 1            2            3            4            5            6
<=20   0.93 / 0.93  0.86 / 0.86  0.72 / 0.72  0.51 / 0.51  0.38 / 0.38  0.27 / 0.27
21-40  0.92 / 0.92  0.88 / 0.88  0.78 / 0.78  0.67 / 0.67  0.50 / 0.50  0.47 / 0.47
41-80  1.00 / 1.00  0.96 / 0.96  0.91 / 0.91  0.81 / 0.81  0.78 / 0.78  0.72 / 0.72
>80    0.77 / 0.84  0.85 / 0.92  0.87 / 0.87  0.85 / 0.85  0.87 / 0.87  0.85 / 0.85
K=20:
                 1            2      

K=30:
                 1            2            3            4            5            6
<=20   0.94 / 0.94  0.65 / 0.65  0.42 / 0.42  0.31 / 0.31  0.29 / 0.29  0.18 / 0.18
21-40  0.92 / 0.92  0.70 / 0.70  0.37 / 0.37  0.26 / 0.26  0.19 / 0.19  0.20 / 0.20
41-80  1.00 / 1.00  0.86 / 0.86  0.65 / 0.65  0.43 / 0.43  0.40 / 0.40  0.37 / 0.37
>80    0.93 / 0.93  0.92 / 0.92  0.87 / 0.87  0.65 / 0.65  0.60 / 0.60  0.48 / 0.49

C1: share of pages where E>=2 is unreachable but E>=1 is not (dense ROIs), and where even E>=1 is unreachable, by iteration:
          unreach E2 only             unreach E1            
K                      10    20    30         10    20    30
iteration                                                   
1                   0.082 0.082 0.061      0.020 0.000 0.000
2                   0.061 0.061 0.061      0.020 0.000 0.000
3                   0.061 0.061 0.041      0.000 0.000 0.000
4                   0.061 0.041 0.041      0.000 0.000 0.000
5                   0


page 1 per ROI, central setting -- ROIs whose mean P@K over 6 clicks is below break-even (p < p*) and below target (p < 2 p*):
  C1             K=10: below break-even 4/49, below target 13/49 (of which E>=2 unreachable: 5); below target among reachable ROIs 8/44
  C1             K=20: below break-even 2/49, below target 14/49 (of which E>=2 unreachable: 4); below target among reachable ROIs 10/45
  C1             K=30: below break-even 3/49, below target 13/49 (of which E>=2 unreachable: 3); below target among reachable ROIs 10/46
  od_unfiltered  K=10: below break-even 7/49, below target 20/49 (of which E>=2 unreachable: 5); below target among reachable ROIs 15/44
  od_unfiltered  K=20: below break-even 6/49, below target 16/49 (of which E>=2 unreachable: 4); below target among reachable ROIs 12/45
  od_unfiltered  K=30: below break-even 5/49, below target 18/49 (of which E>=2 unreachable: 3); below target among reachable ROIs 15/46
  tm_unfiltered  K=10: below break-even 6/49, below

## Across the grid, regime B, and the density-dependent T

For each of the 9 (v, T) settings, the grid recomputes every page's E and E_max from `n`, `TP`, `r` and `R0`. It then
reports the headline shares (page 1 exactly, and all simulated pages), the ROIs below break-even or target at page 1,
and the median minutes saved per ROI over all 6 pages.

Regime B shares cover v × c_saved. The density-T sensitivity uses `T + r·c_saved` at the central v and T.

In [15]:
def session_minutes(frame, saved_col, E):
    """
        Minutes saved per ROI under three stop rules: every page; the oracle stop before the first page with E < 1; the observable stop after the first zero-TP page.

        frame (pd.DataFrame): page rows of one or more (arm, variant, K) groups, with arm, variant, K, file_name, order, iteration, tp.
        saved_col (str): the per-page seconds-saved column.
        E (np.ndarray): each row's E, which drives the oracle stop.

        Returns pd.DataFrame: per (arm, variant, K, file_name), minutes saved under 'all', 'oracle' and 'observable', each the mean over the ROI's orders.
    """
    d = frame[['arm', 'variant', 'K', 'file_name', 'order', 'iteration', 'tp', saved_col]].assign(E=np.asarray(E)).sort_values(['arm', 'variant', 'K', 'file_name', 'order', 'iteration'], kind='mergesort')
    keys = ['arm', 'variant', 'K', 'file_name', 'order']
    low = (d['E'] < 1).astype(int)
    zero = (d['tp'] == 0).astype(int)
    d['oracle_in'] = d.assign(x=low).groupby(keys, sort=False)['x'].cumsum() == 0
    d['obs_in'] = (d.assign(x=zero).groupby(keys, sort=False)['x'].cumsum() - zero) == 0
    d['s_all'] = d[saved_col]
    d['s_oracle'] = d[saved_col] * d['oracle_in']
    d['s_obs'] = d[saved_col] * d['obs_in']
    per_session = d.groupby(keys, sort=False)[['s_all', 's_oracle', 's_obs']].sum()
    per_roi = per_session.groupby(level=['arm', 'variant', 'K', 'file_name'], sort=False).mean() / 60.0
    return per_roi.rename(columns={'s_all': 'all', 's_oracle': 'oracle', 's_obs': 'observable'}).reset_index()


grows = []
for v, tmin in GRID:
    T = 60.0 * tmin
    A = regime_a(PAGES['n'], PAGES['tp'], PAGES['r'], PAGES['R0'], v, T)
    G = PAGES[['arm', 'variant', 'K', 'file_name', 'order', 'iteration', 'tp', 'bin']].assign(e1=(A['E'] >= 1).astype(float), e2=(A['E'] >= 2).astype(float), e1r=np.where(A['E_max'] >= 1, (A['E'] >= 1).astype(float), np.nan), e2r=np.where(A['E_max'] >= 2, (A['E'] >= 2).astype(float), np.nan), sp=A['saved_page'], ss=A['saved_session'])
    mins_p = session_minutes(G, 'sp', A['E'])
    mins_s = session_minutes(G, 'ss', A['E'])
    for (arm, variant), _ in G.groupby(['arm', 'variant'], sort=False):
        p1 = page1_table(arm, v, T, variant)
        for k in KS:
            sub = G[(G['arm'] == arm) & (G['variant'] == variant) & (G['K'] == k)]
            per = sub.groupby('file_name')[['e1', 'e2', 'e1r', 'e2r']].mean()
            q1 = p1[p1['K'] == k]
            s1 = STANDALONE[(STANDALONE['arm'] == arm) & (STANDALONE['variant'] == variant) & (STANDALONE['K'] == k)]
            A1 = regime_a(s1['n'], s1['tp'], s1['r'], s1['R0'], v, T)
            per1 = s1.assign(e1=(A1['E'] >= 1).astype(float), e2=(A1['E'] >= 2).astype(float), e2r=np.where(A1['E_max'] >= 2, (A1['E'] >= 2).astype(float), np.nan)).groupby('file_name')[['e1', 'e2', 'e2r']].mean()
            mp = mins_p[(mins_p['arm'] == arm) & (mins_p['variant'] == variant) & (mins_p['K'] == k)]
            ms = mins_s[(mins_s['arm'] == arm) & (mins_s['variant'] == variant) & (mins_s['K'] == k)]
            grows.append(dict(regime='A', arm=arm, variant=variant, K=k, v=v, T_min=tmin, c_saved=np.nan, central=(v, tmin) == (V0, T0_MIN), page1_share_E1=per1['e1'].mean(), page1_share_E2=per1['e2'].mean(), page1_share_E2_reachable=per1['e2r'].mean(), page1_rois_below_break_even=int(q1['below_break_even'].sum()), page1_rois_below_target=int(q1['below_target'].sum()), page1_rois_E2_unreachable=int((q1['reach'] != 'reachable').sum()), all_share_E1=per['e1'].mean(), all_share_E1_reachable=per['e1r'].mean(), all_share_E2=per['e2'].mean(), all_share_E2_reachable=per['e2r'].mean(), min_saved_page_sum_all_median=mp['all'].median(), min_saved_session_all_median=ms['all'].median(), min_saved_session_observable_median=ms['observable'].median(), rois_net_saving_session_all=float((ms['all'] > 0).mean())))
for v in V_GRID:
    for cs in C_SAVED_GRID:
        B = regime_b(PAGES['n'], PAGES['tp'], PAGES['r'], v, cs)
        B1 = regime_b(STANDALONE['n'], STANDALONE['tp'], STANDALONE['r'], v, cs)
        S1 = STANDALONE.assign(eb=(B1['E'] >= 1).astype(float))
        G = PAGES[['arm', 'variant', 'K', 'file_name', 'iteration']].assign(eb=(B['E'] >= 1).astype(float), ebr=np.where(B['reachable'], (B['E'] >= 1).astype(float), np.nan), sb=B['saved'])
        for (arm, variant, k), sub in G.groupby(['arm', 'variant', 'K'], sort=False):
            per = sub.groupby('file_name')[['eb', 'ebr', 'sb']].mean()
            per1 = S1[(S1['arm'] == arm) & (S1['variant'] == variant) & (S1['K'] == k)].groupby('file_name')['eb'].mean()
            grows.append(dict(regime='B', arm=arm, variant=variant, K=k, v=v, T_min=np.nan, c_saved=cs, central=(v, cs) == (V0, C_SAVED0), p_star_full_page=(k * v + C_BOX + L_WAIT) / (k * cs), page1_share_E1=per1.mean(), all_share_E1=per['eb'].mean(), all_share_E1_reachable=per['ebr'].mean(), mean_saved_per_page_s=per['sb'].mean()))
for cs in C_SAVED_GRID:
    D = regime_a_density(PAGES['n'], PAGES['tp'], PAGES['r'], V0, T0, cs)
    D1 = regime_a_density(STANDALONE['n'], STANDALONE['tp'], STANDALONE['r'], V0, T0, cs)
    S1 = STANDALONE.assign(e1=(D1['E'] >= 1).astype(float), e2=(D1['E'] >= 2).astype(float))
    G = PAGES[['arm', 'variant', 'K', 'file_name', 'iteration']].assign(e1=(D['E'] >= 1).astype(float), e2=(D['E'] >= 2).astype(float), e2r=np.where(D['E_max'] >= 2, (D['E'] >= 2).astype(float), np.nan), unr=(D['E_max'] < 2).astype(float))
    for (arm, variant, k), sub in G.groupby(['arm', 'variant', 'K'], sort=False):
        per = sub.groupby('file_name')[['e1', 'e2', 'e2r', 'unr']].mean()
        per1 = S1[(S1['arm'] == arm) & (S1['variant'] == variant) & (S1['K'] == k)].groupby('file_name')[['e1', 'e2']].mean()
        grows.append(dict(regime='A_density_T', arm=arm, variant=variant, K=k, v=V0, T_min=T0_MIN, c_saved=cs, central=cs == C_SAVED0, page1_share_E1=per1['e1'].mean(), page1_share_E2=per1['e2'].mean(), all_share_E1=per['e1'].mean(), all_share_E2=per['e2'].mean(), all_share_E2_reachable=per['e2r'].mean(), share_pages_E2_unreachable=per['unr'].mean()))
GRIDT = pd.DataFrame(grows)

ga = GRIDT[(GRIDT['regime'] == 'A') & (GRIDT['variant'] == 'recomputed')]
for k in KS:
    print(f'\nregime A across the grid, K={k}: page-1 shares (exact, standalone) and all-pages shares, E>=2 reachable-only, ROIs below target at page 1, median minutes saved per ROI (session accounting)')
    t = ga[ga['K'] == k].assign(setting=lambda d: [f'v{x:g} T{y}' for x, y in zip(d['v'], d['T_min'])]).pivot(index='setting', columns='arm', values='all_share_E2_reachable')[SIM_ARMS]
    t1 = ga[ga['K'] == k].assign(setting=lambda d: [f'v{x:g} T{y}' for x, y in zip(d['v'], d['T_min'])]).pivot(index='setting', columns='arm', values='page1_share_E2')[SIM_ARMS]
    bt = ga[ga['K'] == k].assign(setting=lambda d: [f'v{x:g} T{y}' for x, y in zip(d['v'], d['T_min'])]).pivot(index='setting', columns='arm', values='page1_rois_below_target')[SIM_ARMS]
    mm = ga[ga['K'] == k].assign(setting=lambda d: [f'v{x:g} T{y}' for x, y in zip(d['v'], d['T_min'])]).pivot(index='setting', columns='arm', values='min_saved_session_all_median')[SIM_ARMS]
    print(pd.concat({'page1 E>=2': t1, 'all E>=2 reach': t, 'p1 ROIs<2p*': bt, 'med min (session)': mm}, axis=1).to_string(float_format=lambda v: f'{v:.2f}'))
gb = GRIDT[(GRIDT['regime'] == 'B') & (GRIDT['variant'] == 'recomputed')]
print('\nregime B (a full pass is mandatory): share of pages with E_B >= 1, page 1 / all pages, and p*_B for a full page:')
for k in KS:
    t = gb[gb['K'] == k].assign(setting=lambda d: [f'v{x:g} cs{y:g} p*B={z:.2f}' for x, y, z in zip(d['v'], d['c_saved'], d['p_star_full_page'])])
    print(f'K={k}:')
    print(pd.concat({'page1': t.pivot(index='setting', columns='arm', values='page1_share_E1')[SIM_ARMS], 'all pages': t.pivot(index='setting', columns='arm', values='all_share_E1')[SIM_ARMS]}, axis=1).to_string(float_format=lambda v: f'{v:.2f}'))
gd = GRIDT[(GRIDT['regime'] == 'A_density_T') & (GRIDT['variant'] == 'recomputed')]
print('\nsensitivity, T grows with density (T_eff = T + r c_saved; v 2 s, T 10 min): page-1 and all-pages share with E>=2, and share of pages where E>=2 is unreachable:')
print(gd.pivot_table(index=['K', 'c_saved'], columns='arm', values=['page1_share_E2', 'all_share_E2', 'share_pages_E2_unreachable'])[[(m, a) for m in ('page1_share_E2', 'all_share_E2', 'share_pages_E2_unreachable') for a in SIM_ARMS]].to_string(float_format=lambda v: f'{v:.2f}'))


regime A across the grid, K=10: page-1 shares (exact, standalone) and all-pages shares, E>=2 reachable-only, ROIs below target at page 1, median minutes saved per ROI (session accounting)
           page1 E>=2                                  all E>=2 reach                                  p1 ROIs<2p*                                   med min (session)                                  
arm                C1 od_unfiltered tm_unfiltered   od             C1 od_unfiltered tm_unfiltered   od          C1 od_unfiltered tm_unfiltered    od                C1 od_unfiltered tm_unfiltered    od
setting                                                                                                                                                                                                 
v1.5 T10.0       0.80          0.67          0.70 0.72           0.63          0.57          0.55 0.58        9.00         16.00         12.00 13.00              2.46          1.94          2.00  1.97
v1.5 T2

## Time saved per ROI, at the central setting

Each figure is the mean over a ROI's 20 orders, in minutes. It is computed under three stop rules and three
accountings:
- **`page_sum`** is the prompt's sum of `(E − 1)·cost` over pages, with each page valued at T/r_j.
- **`session`** fixes r at the ROI total: `ΣTP·T/R0 − Σcost`. This is the coherent "against no tool" figure, with
  seeds treated as found at the no-tool rate.
- **`session_seed`** is `session` minus the extra cost of hand-finding later seeds in a depleted ROI.

The tables report the median and IQR over ROIs, and the share of ROIs with a net saving. They then give paired
ROI-level differences, best method minus each comparison arm, with a bootstrap CI.

Savings are linear in T, so each ROI also has a closed-form **break-even pass time** `T_break`. A full manual pass of
the ROI must take longer than `T_break` for the tool to save time:
- **page 1:** `T_break = target·r·cost/(K·p)`;
- **session:** `T_break = R0·Σcost/ΣTP`, for all six pages and for the observable stop;
- **with the seed premium:** `T_break = Σcost/(ΣTP/R0 − Σ(1/r_pre − 1/R0))`.

`T_break` turns the unmeasured T into one number to time against a real pass.

In [16]:
MIN = {}
for acc, col in (('page_sum', 'saved_page_s'), ('session', 'saved_session_s'), ('session_seed', 'saved_session_seed_s')):
    MIN[acc] = session_minutes(PAGES, col, PAGES['E'].to_numpy()).assign(accounting=acc)
MINUTES = pd.concat(MIN.values(), ignore_index=True)
mrows = []
for (acc, arm, variant, k), g in MINUTES.groupby(['accounting', 'arm', 'variant', 'K'], sort=False):
    for rule in ('all', 'oracle', 'observable'):
        x = g[rule]
        mrows.append(dict(accounting=acc, arm=arm, variant=variant, K=k, stop=rule, median=x.median(), q25=x.quantile(0.25), q75=x.quantile(0.75), mean=x.mean(), share_rois_net_saving=float((x > 0).mean()), n_roi=len(x)))
MSUM = pd.DataFrame(mrows)
collect('minutes_saved', 'minutes saved per ROI (median, IQR; mean)', MSUM.rename(columns={'stop': 'group'}))
for acc in ('session', 'page_sum', 'session_seed'):
    print(f'\nminutes saved per ROI, central, accounting "{acc}": median [IQR]; share of ROIs with a net saving')
    t = MSUM[(MSUM['accounting'] == acc) & (MSUM['variant'] == 'recomputed')].assign(cell=lambda d: [f'{m:.1f} [{a:.1f}, {b:.1f}] {100 * s:.0f}%' for m, a, b, s in zip(d['median'], d['q25'], d['q75'], d['share_rois_net_saving'])])
    print(t.pivot_table(index=['K', 'stop'], columns='arm', values='cell', aggfunc='first')[SIM_ARMS].to_string())

prows = []
for acc in ('session', 'page_sum', 'session_seed'):
    m = MINUTES[(MINUTES['accounting'] == acc) & (MINUTES['variant'] == 'recomputed')]
    for k in KS:
        for rule in ('all', 'oracle', 'observable'):
            best = m[(m['arm'] == BEST) & (m['K'] == k)].set_index('file_name')[rule]
            for other in COMPARISON_ARMS:
                prows.append(dict(accounting=acc, K=k, stop=rule, contrast=f'{BEST} - {other}', **paired(best, m[(m['arm'] == other) & (m['K'] == k)].set_index('file_name')[rule])))
PAIRED = pd.DataFrame(prows)
collect('minutes_saved_paired', 'minutes saved per ROI, best minus arm (mean of paired ROI differences)', PAIRED.rename(columns={'stop': 'group'}))
print('\npaired ROI-level difference in minutes saved, C1 (in-sample) minus each arm, accounting "session", central: mean [95% bootstrap CI], W/L/T')
t = PAIRED[PAIRED['accounting'] == 'session'].assign(cell=lambda d: [fmt_ci(a, b, c, 1, 2) + f' {w}/{l}/{t_}' for a, b, c, w, l, t_ in zip(d['mean'], d['ci_lo'], d['ci_hi'], d['wins'], d['losses'], d['ties'])])
print(t.pivot_table(index=['K', 'stop'], columns='contrast', values='cell', aggfunc='first').to_string())
print('\nthe same, accounting "page_sum" (the prompt\'s):')
t = PAIRED[PAIRED['accounting'] == 'page_sum'].assign(cell=lambda d: [fmt_ci(a, b, c, 1, 2) + f' {w}/{l}/{t_}' for a, b, c, w, l, t_ in zip(d['mean'], d['ci_lo'], d['ci_hi'], d['wins'], d['losses'], d['ties'])])
print(t.pivot_table(index=['K', 'stop'], columns='contrast', values='cell', aggfunc='first').to_string())

# break-even pass time T: session savings are linear in T, so each ROI has a closed-form T above which the tool saves time
d = PAGES[['arm', 'variant', 'K', 'file_name', 'order', 'iteration', 'tp', 'R0', 'cost_s', 'seed_already_found', 'r_pre_box']].sort_values(['arm', 'variant', 'K', 'file_name', 'order', 'iteration'], kind='mergesort')
skeys = ['arm', 'variant', 'K', 'file_name', 'order']
z = (d['tp'] == 0).astype(int)
d['obs_in'] = (d.assign(x=z).groupby(skeys, sort=False)['x'].cumsum() - z) == 0
d['prem_coef'] = np.where(d['seed_already_found'], 0.0, 1.0 / d['r_pre_box'] - 1.0 / d['R0'])
d['tp_obs'], d['cost_obs'] = d['tp'] * d['obs_in'], d['cost_s'] * d['obs_in']
tb = d.groupby(skeys, sort=False).agg(tp=('tp', 'sum'), cost=('cost_s', 'sum'), prem=('prem_coef', 'sum'), tp_obs=('tp_obs', 'sum'), cost_obs=('cost_obs', 'sum'), R0=('R0', 'first')).groupby(level=['arm', 'variant', 'K', 'file_name'], sort=False).mean()
with np.errstate(divide='ignore', invalid='ignore'):
    tb['T_break_session_all'] = np.where(tb['tp'] > 0, tb['R0'] * tb['cost'] / tb['tp'], np.inf) / 60.0
    tb['T_break_session_observable'] = np.where(tb['tp_obs'] > 0, tb['R0'] * tb['cost_obs'] / tb['tp_obs'], np.inf) / 60.0
    den = tb['tp'] / tb['R0'] - tb['prem']
    tb['T_break_session_seed_all'] = np.where(den > 0, tb['cost'] / den, np.inf) / 60.0
TBREAK = tb.reset_index()
p1t = []
for arm in SIM_ARMS:
    for variant in variants_for(arm):
        s1 = STANDALONE[(STANDALONE['arm'] == arm) & (STANDALONE['variant'] == variant)].groupby(['K', 'file_name']).agg(p=('p', 'mean'), r=('r', 'first'))
        cost1 = s1.index.get_level_values('K').to_numpy() * V0 + C_BOX + L_WAIT
        with np.errstate(divide='ignore'):
            for tgt in (1, 2):
                s1[f'T_break_page1_E{tgt}'] = np.where(s1['p'] > 0, tgt * s1['r'] * cost1 / (s1.index.get_level_values('K').to_numpy() * s1['p']), np.inf) / 60.0
        p1t.append(s1.reset_index().assign(arm=arm, variant=variant))
TBREAK = TBREAK.merge(pd.concat(p1t, ignore_index=True)[['arm', 'variant', 'K', 'file_name', 'T_break_page1_E1', 'T_break_page1_E2']], on=['arm', 'variant', 'K', 'file_name'], validate='one_to_one')
TB_COLS = ['T_break_page1_E1', 'T_break_page1_E2', 'T_break_session_all', 'T_break_session_observable', 'T_break_session_seed_all']
tbrows = []
for (arm, variant, k), g in TBREAK.groupby(['arm', 'variant', 'K'], sort=False):
    for col in TB_COLS:
        x = g[col]
        tbrows.append(dict(arm=arm, variant=variant, K=k, group=col, median=x.median(), q25=x.quantile(0.25), q75=x.quantile(0.75), share_le_5=float((x <= 5).mean()), share_le_10=float((x <= 10).mean()), share_le_20=float((x <= 20).mean()), n_roi=len(x)))
TBSUM = pd.DataFrame(tbrows)
collect('T_break', 'break-even manual-pass minutes per ROI (v 2 s): median, IQR, share of ROIs at or below 5/10/20 min', TBSUM)
print('\nbreak-even manual-pass time T per ROI, in minutes (v 2 s, c 5 s, L 2 s): the tool saves time only if a full manual pass of the ROI takes longer than this. median [IQR]; share of ROIs with T_break <= 10 min')
t = TBSUM[TBSUM['variant'] == 'recomputed'].assign(cell=lambda q: [f'{m:.1f} [{a:.1f}, {b:.1f}] {100 * s10:.0f}%' for m, a, b, s10 in zip(q['median'], q['q25'], q['q75'], q['share_le_10'])])
print(t.pivot_table(index=['K', 'group'], columns='arm', values='cell', aggfunc='first').reindex(pd.MultiIndex.from_product([KS, TB_COLS]))[SIM_ARMS].to_string())

ms = MINUTES[(MINUTES['accounting'] == 'session') & (MINUTES['arm'] == BEST) & (MINUTES['variant'] == 'recomputed')].merge(ROI_INFO[['file_name', 'bin']], on='file_name')
print('\nC1, session accounting, all pages: median minutes saved per ROI by density bin')
print(ms.pivot_table(index='bin', columns='K', values='all', aggfunc='median').reindex(DENSITY_LABELS).to_string(float_format=lambda v: f'{v:.1f}'))


minutes saved per ROI, central, accounting "session": median [IQR]; share of ROIs with a net saving
arm                             C1         od_unfiltered         tm_unfiltered                    od
K  stop                                                                                             
10 all          2.0 [0.7, 3.3] 88%    1.4 [0.2, 3.0] 84%    1.5 [0.1, 2.4] 82%    1.5 [0.3, 3.0] 86%
   observable   2.0 [1.2, 3.3] 90%    1.5 [0.6, 3.0] 88%    1.5 [0.6, 2.7] 84%    1.5 [0.6, 3.0] 88%
   oracle       2.2 [1.4, 3.5] 94%    1.8 [0.8, 3.4] 92%    1.7 [0.7, 3.2] 90%    1.8 [0.9, 3.3] 92%
20 all          1.2 [0.1, 2.2] 76%   0.9 [-0.3, 2.1] 69%   0.8 [-0.4, 1.9] 67%   1.0 [-0.2, 2.0] 67%
   observable   1.7 [0.7, 2.9] 84%    1.5 [0.2, 2.7] 78%   1.3 [-0.0, 2.4] 71%    1.5 [0.2, 2.8] 80%
   oracle       2.3 [1.5, 3.6] 96%    1.9 [1.0, 3.5] 90%    1.8 [0.6, 3.1] 94%    1.9 [1.0, 3.5] 90%
30 all         0.1 [-0.9, 1.4] 57%  -0.1 [-1.0, 1.3] 49%  -0.1 [-1.1, 1.3] 45%  -0.0 [-1.1,


paired ROI-level difference in minutes saved, C1 (in-sample) minus each arm, accounting "session", central: mean [95% bootstrap CI], W/L/T
contrast                         C1 - od         C1 - od_unfiltered        C1 - tm_unfiltered
K  stop                                                                                      
10 all          0.30 [0.18, 0.41] 41/7/1   0.31 [0.17, 0.44] 41/8/0  0.51 [0.34, 0.66] 40/8/1
   observable   0.27 [0.13, 0.40] 38/9/2   0.28 [0.11, 0.43] 39/9/1  0.50 [0.31, 0.67] 40/7/2
   oracle      0.26 [0.12, 0.40] 35/11/3  0.28 [0.11, 0.43] 37/10/2  0.48 [0.29, 0.66] 40/6/3
20 all          0.22 [0.13, 0.30] 42/6/1   0.19 [0.10, 0.29] 39/9/1  0.39 [0.27, 0.51] 41/7/1
   observable   0.20 [0.09, 0.31] 41/8/0   0.19 [0.07, 0.32] 40/9/0  0.49 [0.35, 0.64] 40/9/0
   oracle      0.21 [0.10, 0.32] 36/12/1  0.21 [0.08, 0.34] 38/10/1  0.52 [0.37, 0.67] 40/8/1
30 all          0.11 [0.04, 0.18] 40/9/0   0.12 [0.04, 0.21] 40/9/0  0.23 [0.14, 0.32] 42/7/0
   observable 


break-even manual-pass time T per ROI, in minutes (v 2 s, c 5 s, L 2 s): the tool saves time only if a full manual pass of the ROI takes longer than this. median [IQR]; share of ROIs with T_break <= 10 min
arm                                              C1         od_unfiltered         tm_unfiltered                    od
10 T_break_page1_E1              2.9 [1.8, 5.6] 92%    3.7 [1.9, 7.1] 86%    3.7 [2.2, 5.9] 88%    3.5 [1.9, 6.4] 88%
   T_break_page1_E2             5.7 [3.6, 11.3] 73%   7.5 [3.7, 14.3] 59%   7.5 [4.5, 11.9] 57%   7.0 [3.9, 12.8] 63%
   T_break_session_all           5.8 [4.5, 8.0] 88%    6.5 [4.8, 9.3] 84%    6.4 [5.3, 9.5] 82%    6.5 [4.8, 9.2] 86%
   T_break_session_observable    5.3 [3.9, 6.7] 90%    5.5 [4.1, 8.1] 88%    5.9 [4.5, 8.3] 84%    5.7 [4.1, 7.9] 88%
   T_break_session_seed_all     6.8 [5.3, 10.5] 73%   7.6 [5.5, 10.5] 71%   7.2 [6.3, 11.3] 67%   7.5 [5.7, 10.2] 71%
20 T_break_page1_E1              3.2 [1.9, 5.2] 96%    3.3 [2.0, 5.6] 88%    4.1 [2.4

## The two rankings within an iteration: recomputed against the full-pool order (C1)

`C1`'s ranks are relative to the pool. Removing candidates can therefore change the order of the remaining ones. The
**fullpool** variant ignores that. The table below gives the paired ROI-level difference in page precision,
recomputed minus fullpool, by iteration.

In [17]:
c1 = PAGES[(PAGES['arm'] == BEST) & PAGES['shown']]
pv = c1.pivot_table(index=['K', 'file_name', 'order', 'iteration'], columns='variant', values='p')
same_n = c1.pivot_table(index=['K', 'file_name', 'order', 'iteration'], columns='variant', values='n')
vrows = []
for (k, it), g in pv.groupby(level=['K', 'iteration']):
    per = g.groupby(level='file_name').mean()
    vrows.append(dict(K=k, iteration=it, **paired(per['recomputed'], per['fullpool']), share_pages_differ=float((g['recomputed'] != g['fullpool']).mean())))
VAR = pd.DataFrame(vrows)
collect('variant_recomputed_minus_fullpool', 'C1 page precision, recomputed minus fullpool', VAR)
print('C1 page precision, recomputed minus fullpool (paired over ROIs): mean [95% CI], W/L/T, and the share of pages whose precision differs')
print(VAR.assign(cell=lambda d: [fmt_ci(a, b, c, 1, 4) + f' {w}/{l}/{t_}' for a, b, c, w, l, t_ in zip(d['mean'], d['ci_lo'], d['ci_hi'], d['wins'], d['losses'], d['ties'])]).pivot(index='iteration', columns='K', values='cell').to_string())
print(VAR.pivot(index='iteration', columns='K', values='share_pages_differ').to_string(float_format=lambda v: f'{v:.3f}'))
mv = MINUTES[(MINUTES['accounting'] == 'session') & (MINUTES['arm'] == BEST)]
for k in KS:
    a = mv[(mv['variant'] == 'recomputed') & (mv['K'] == k)].set_index('file_name')['all']
    b = mv[(mv['variant'] == 'fullpool') & (mv['K'] == k)].set_index('file_name')['all']
    d = paired(a, b)
    print(f'K={k}: minutes saved (session, all pages), recomputed minus fullpool: {fmt_ci(d["mean"], d["ci_lo"], d["ci_hi"], 1, 3)}')

C1 page precision, recomputed minus fullpool (paired over ROIs): mean [95% CI], W/L/T, and the share of pages whose precision differs
K                                          10                                  20                                   30
iteration                                                                                                             
1              0.0000 [0.0000, 0.0000] 0/0/49      0.0000 [0.0000, 0.0000] 0/0/49       0.0000 [0.0000, 0.0000] 0/0/49
2          -0.0032 [-0.0101, 0.0030] 17/22/10   0.0004 [-0.0018, 0.0027] 17/19/13  -0.0023 [-0.0045, -0.0003] 10/22/17
3            0.0006 [-0.0073, 0.0097] 20/22/7  -0.0012 [-0.0050, 0.0022] 16/23/10    0.0002 [-0.0018, 0.0025] 18/15/16
4           0.0020 [-0.0047, 0.0089] 20/19/10  -0.0018 [-0.0066, 0.0028] 15/21/13    0.0014 [-0.0003, 0.0032] 21/14/14
5          -0.0012 [-0.0082, 0.0059] 17/22/10  -0.0014 [-0.0043, 0.0016] 14/22/13    0.0008 [-0.0013, 0.0031] 22/12/15
6          -0.0002 [-0.0069, 0.00

## Diagnostics: why later pages fall

The click at iteration j is a uniformly random one of the ROI's 6. Its standalone precision `p_fresh` therefore equals
page 1's in expectation, and any decline across iterations comes from the session's removals. Each page splits into
three factors:
- **depletion**: `r`, the unfound mitoses;
- **coverage, or pool exhaustion**: `mitoses_reachable / r`, the share of the unfound mitoses that are still within R
  of a remaining candidate in this click's saved, 100-peak-capped pool;
- **ranking efficiency**: `TP / min(n, mitoses_reachable)`.

Then `p = ceiling × efficiency`, where `ceiling = min(n, reachable)/n`.

The section also reports:
- under-filled and skipped pages;
- how many seeds were already found by the tool;
- removals;
- hidden mitoses.

In [18]:
dcols = ['p', 'p_fresh', 'r', 'mitoses_reachable', 'coverage', 'ceiling', 'efficiency', 'n_remaining', 'under', 'skipped', 'seed_found_it2plus', 'n_removed_shown', 'n_removed_found', 'hidden_mitoses', 'hidden_near_shown']
PRIM2 = PRIM.assign(skipped=PRIM['skipped'].astype(float), hidden_any=(PRIM['hidden_mitoses'] > 0).astype(float))
DIAG = []
for col in dcols + ['hidden_any']:
    s = summarise(PRIM2 if col not in ('p', 'p_fresh', 'ceiling', 'efficiency') else PRIM2[PRIM2['shown']], col, ['arm', 'K', 'iteration'])
    DIAG.append(s.assign(metric=col))
    collect('diagnostics', col, s)
    collect('diagnostics_bin', col, summarise(PRIM2 if col not in ('p', 'p_fresh', 'ceiling', 'efficiency') else PRIM2[PRIM2['shown']], col, ['arm', 'K', 'iteration', 'bin']))
DIAG = pd.concat(DIAG, ignore_index=True)
for arm in SIM_ARMS:
    for k in KS:
        t = DIAG[(DIAG['arm'] == arm) & (DIAG['K'] == k)].pivot(index='iteration', columns='metric', values='mean')
        print(f'\n{arm}, K={k}: mean over ROIs per iteration')
        print(t[['p', 'p_fresh', 'r', 'mitoses_reachable', 'coverage', 'ceiling', 'efficiency', 'n_remaining', 'under', 'skipped', 'seed_found_it2plus', 'hidden_mitoses', 'hidden_any', 'hidden_near_shown']].to_string(float_format=lambda v: f'{v:.3f}'))

sess = PRIM.groupby(['arm', 'K', 'file_name', 'order'])['iteration'].max()
ended = (sess < N_ITER).groupby(level=['arm', 'K']).mean()
print('\nshare of sessions that ended before 6 pages because every mitosis was found (r = 0):')
print(ended.unstack('K').to_string(float_format=lambda v: f'{v:.3f}'))
# what a 6-page session harvests: C1's later pages fall faster because its early pages took the same pool's mitoses sooner
HARV = PRIM.groupby(['arm', 'K', 'file_name', 'order']).agg(tp=('tp', 'sum'), shown=('n', 'sum'), pages=('iteration', 'size')).groupby(level=['arm', 'K', 'file_name']).mean().reset_index()
hrows = []
for k in KS:
    h = HARV[HARV['K'] == k]
    best = h[h['arm'] == BEST].set_index('file_name')['tp']
    for arm in SIM_ARMS:
        m, lo, hi, n = boot_ci(h[h['arm'] == arm]['tp'].to_numpy())
        hrows.append(dict(K=k, arm=arm, contrast='', mean=m, ci_lo=lo, ci_hi=hi, n_roi=n))
    for other in COMPARISON_ARMS:
        hrows.append(dict(K=k, arm=BEST, contrast=f'{BEST} - {other}', **paired(best, h[h['arm'] == other].set_index('file_name')['tp'])))
HARVEST = pd.DataFrame(hrows)
collect('session_harvest', 'mitoses confirmed per 6-page session (mean over orders, then ROIs)', HARVEST)
print('\nmitoses confirmed per 6-page session (seeds excluded), mean over ROIs [95% CI]; and C1 (in-sample) minus each arm, paired over ROIs, W/L/T:')
for k in KS:
    hk = HARVEST[HARVEST['K'] == k]
    print(f'  K={k}: ' + '; '.join(f'{x.arm} {fmt_ci(x.mean, x.ci_lo, x.ci_hi, 1, 2)}' for x in hk[hk['contrast'] == ''].itertuples()))
    print('         ' + '; '.join(f'{x.contrast} {fmt_ci(x.mean, x.ci_lo, x.ci_hi, 1, 2)} {int(x.wins)}/{int(x.losses)}/{int(x.ties)}' for x in hk[hk['contrast'] != ''].itertuples()))
cum = PRIM.sort_values('iteration').assign(cum_tp=lambda d: d.groupby(['arm', 'K', 'file_name', 'order'])['tp'].cumsum())
ct = cum.groupby(['arm', 'K', 'iteration', 'file_name'])['cum_tp'].mean().groupby(level=['arm', 'K', 'iteration']).mean().unstack('iteration')
print('\ncumulative mitoses confirmed after each page (mean over ROIs; sessions that ended early carry their total):')
print(ct.to_string(float_format=lambda v: f'{v:.2f}'))

# the decline split into its factors, each relative to page 1 (ratios of ROI means)
rat = []
for arm in SIM_ARMS:
    for k in KS:
        t = DIAG[(DIAG['arm'] == arm) & (DIAG['K'] == k)].pivot(index='iteration', columns='metric', values='mean')
        for it in range(2, N_ITER + 1):
            rat.append(dict(arm=arm, K=k, iteration=it, depletion_r=t.loc[it, 'r'] / t.loc[1, 'r'], reachable=t.loc[it, 'mitoses_reachable'] / t.loc[1, 'mitoses_reachable'], coverage=t.loc[it, 'coverage'] / t.loc[1, 'coverage'], ceiling=t.loc[it, 'ceiling'] / t.loc[1, 'ceiling'], efficiency=t.loc[it, 'efficiency'] / t.loc[1, 'efficiency'], precision=t.loc[it, 'p'] / t.loc[1, 'p']))
RATIOS = pd.DataFrame(rat)
collect('decline_factors', 'ratio to page 1 (of ROI means)', RATIOS.melt(id_vars=['arm', 'K', 'iteration'], var_name='group', value_name='mean'))
print('\nwhy later pages fall -- each factor at iteration j as a ratio to page 1 (ratios of ROI means). depletion_r = r_j / r_1; coverage = share of the unfound mitoses still reachable from the remaining saved pool (pool exhaustion beyond depletion); efficiency = ranking; precision ~ ceiling x efficiency:')
for k in KS:
    print(f'K={k}:')
    print(RATIOS[RATIOS['K'] == k].set_index(['arm', 'iteration'])[['depletion_r', 'reachable', 'coverage', 'ceiling', 'efficiency', 'precision']].to_string(float_format=lambda v: f'{v:.2f}'))

sb = summarise(PRIM2[PRIM2['iteration'] >= 2], 'seed_found_it2plus', ['arm', 'K', 'bin'])
print('\nshare of iteration >= 2 seeds already found by the tool (those boxes need no search), by density bin:')
print(sb.pivot_table(index=['arm', 'K'], columns='bin', values='mean')[list(DENSITY_LABELS)].to_string(float_format=lambda v: f'{v:.3f}'))


C1, K=10: mean over ROIs per iteration
metric        p  p_fresh      r  mitoses_reachable  coverage  ceiling  efficiency  n_remaining  under  skipped  seed_found_it2plus  hidden_mitoses  hidden_any  hidden_near_shown
iteration                                                                                                                                                                        
1         0.642    0.642 55.776             24.248     0.477    0.904       0.683       96.307  0.000    0.000                 NaN           0.000       0.000              0.000
2         0.477    0.636 48.550             18.326     0.389    0.735       0.605       87.497  0.000    0.000               0.193           0.009       0.009              0.009
3         0.381    0.637 43.113             14.241     0.329    0.620       0.565       79.411  0.000    0.000               0.334           0.029       0.029              0.029
4         0.307    0.636 38.729             10.909     0.271    0.516 


cumulative mitoses confirmed after each page (mean over ROIs; sessions that ended early carry their total):
iteration            1     2     3     4     5     6
arm           K                                     
C1            10  6.42 11.19 15.00 18.07 20.66 22.81
              20 11.01 17.85 22.71 26.32 29.03 31.35
              30 14.47 22.24 27.31 30.34 32.27 33.93
od            10  5.91 10.40 13.60 16.58 19.12 21.32
              20 10.37 16.64 21.36 25.02 27.94 30.28
              30 13.58 21.07 26.28 29.38 31.77 33.15
od_unfiltered 10  5.65 10.12 13.33 16.29 18.88 21.12
              20 10.19 16.34 21.13 24.88 28.14 30.38
              30 13.31 20.86 26.19 29.48 31.99 33.44
tm_unfiltered 10  5.43  9.82 13.31 16.16 18.55 20.82
              20  9.38 15.75 20.34 23.95 27.00 29.63
              30 12.49 19.98 25.31 28.83 31.22 33.11

why later pages fall -- each factor at iteration j as a ratio to page 1 (ratios of ROI means). depletion_r = r_j / r_1; coverage = share of the unfo

## Sensitivity: refilling the pool from the deep list (the two unfiltered arms)

The saved pools are capped at 100 peaks, and a click's pool shares most of its candidates with the other clicks' pools.
In a product that hides shown candidates *before* applying the cap, deeper candidates would take their place.

The **refill** variant models that for `od_unfiltered` and `tm_unfiltered`:
1. It starts from the click's deep list, NOTEBOOK's own input `deep_pool_mo_pad61_49roi_6seed_postD11_top1000.csv`. That
   is 1000 post-NMS detections in score order, and its first `n_pool` rows are the saved pool.
2. It applies the same shown and found removals.
3. It keeps the first `n_pool` survivors.
4. It ranks them by `od` or by score.

Page 1 equals the saved-pool page exactly. This is asserted, and it was also checked in gate 3.

**Approximations:**
- The refilled pool's size is held at the click's `n_pool`.
- The deep list's NMS ran over all peaks, so a removed candidate can still suppress a neighbour. The refill therefore
  slightly under-fills.

`C1` and `od` cannot be refilled, because the deep rows have no refinement features or refusals.

In [19]:
RF = PAGES[PAGES['arm'].isin(REFILL_ARMS)]
rf_prec = summarise(RF[RF['shown']], 'p', ['arm', 'variant', 'K', 'iteration'])
rf_cov = summarise(RF, 'coverage', ['arm', 'variant', 'K', 'iteration'])
rf_e2 = summarise(RF, 'e2', ['arm', 'variant', 'K', 'iteration'])
for name, frame in (('page precision', rf_prec), ('coverage (reachable / r)', rf_cov), ('share of pages with E>=2, central', rf_e2)):
    collect('refill_vs_saved', name, frame)
    print(f'\n{name}: saved pool (recomputed) vs refilled from the deep list, mean over ROIs')
    print(frame.pivot_table(index=['K', 'iteration'], columns=['arm', 'variant'], values='mean').to_string(float_format=lambda v: f'{v:.3f}'))
rows_rf = []
for acc in ('session', 'page_sum'):
    m = MINUTES[(MINUTES['accounting'] == acc) & MINUTES['arm'].isin(REFILL_ARMS)]
    for (arm, k), g in m.groupby(['arm', 'K'], sort=False):
        a = g[g['variant'] == 'refill'].set_index('file_name')['all']
        b = g[g['variant'] == 'recomputed'].set_index('file_name')['all']
        rows_rf.append(dict(accounting=acc, arm=arm, K=k, saved_median=b.median(), refill_median=a.median(), **{f'refill_minus_saved_{x}': y for x, y in paired(a, b).items()}))
RFT = pd.DataFrame(rows_rf)
collect('refill_minutes', 'minutes saved per ROI, all 6 pages: refill minus saved pool', RFT.rename(columns={'refill_minus_saved_mean': 'mean', 'refill_minus_saved_ci_lo': 'ci_lo', 'refill_minus_saved_ci_hi': 'ci_hi', 'refill_minus_saved_n_roi': 'n_roi'}))
print('\nminutes saved per ROI over all 6 pages, central: median with the saved pool vs refilled, and the paired refill - saved difference (mean [95% CI])')
print(RFT.assign(diff=lambda q: [fmt_ci(a, b, c, 1, 2) for a, b, c in zip(q['refill_minus_saved_mean'], q['refill_minus_saved_ci_lo'], q['refill_minus_saved_ci_hi'])])[['accounting', 'arm', 'K', 'saved_median', 'refill_median', 'diff']].to_string(index=False, float_format=lambda v: f'{v:.2f}'))


page precision: saved pool (recomputed) vs refilled from the deep list, mean over ROIs
arm          od_unfiltered        tm_unfiltered       
variant         recomputed refill    recomputed refill
K  iteration                                          
10 1                 0.565  0.565         0.543  0.543
   2                 0.447  0.449         0.438  0.438
   3                 0.322  0.318         0.349  0.349
   4                 0.297  0.296         0.285  0.285
   5                 0.261  0.257         0.240  0.240
   6                 0.216  0.205         0.214  0.214
20 1                 0.510  0.510         0.469  0.469
   2                 0.307  0.309         0.318  0.318
   3                 0.240  0.237         0.230  0.230
   4                 0.188  0.181         0.181  0.181
   5                 0.154  0.150         0.153  0.153
   6                 0.119  0.114         0.126  0.126
30 1                 0.444  0.444         0.416  0.416
   2                 0.252  0.25

In [20]:
# outputs
PAGE_COLS = ['arm', 'variant', 'K', 'file_name', 'domain', 'R0', 'bin', 'order', 'iteration', 'seed_index', 'seed_ann_id', 'seed_already_found', 'r_pre_box', 'r', 'n_base', 'n_removed_shown', 'n_removed_found', 'n_removed', 'n_remaining', 'n', 'tp', 'n_lookalike', 'n_unlabelled', 'skipped', 'mitoses_reachable', 'hidden_mitoses', 'hidden_near_shown', 'p', 'p_fresh', 'cost_s', 'p_star', 'E', 'E_max', 'reach', 'saved_page_s', 'saved_session_s', 'seed_premium_s', 'p_star_B', 'E_B', 'reach_B', 'saved_B_s', 'E_dens']
# every float column here is a closed form of the integer columns (n, tp, r, R0, r_pre_box) at the central setting, so it is written at 10 significant digits to keep the file small
PAGES[PAGE_COLS].to_csv(OUT['pages'], index=False, float_format='%.10g')

per_roi = []
for arm in SIM_ARMS:
    for variant in variants_for(arm):
        p1 = page1_table(arm, V0, T0, variant)
        for k in KS:
            g = PAGES[(PAGES['arm'] == arm) & (PAGES['variant'] == variant) & (PAGES['K'] == k)]
            q = p1[p1['K'] == k].set_index('file_name')
            it_p = g[g['shown']].pivot_table(index='file_name', columns='iteration', values='p', aggfunc='mean')
            for f in ROI_ORDER:
                gf = g[g['file_name'] == f]
                rec = dict(arm=arm, variant=variant, K=k, file_name=f, domain=DOMAIN_OF[f], R0=ROIS[f]['R0'], bin=ROIS[f]['bin'], page1_p=q.loc[f, 'p'], page1_p_star=q.loc[f, 'p_star'], page1_two_p_star=q.loc[f, 'two_p_star'], page1_E_max=q.loc[f, 'E_max'], page1_reach=q.loc[f, 'reach'], page1_below_break_even=bool(q.loc[f, 'below_break_even']), page1_below_target=bool(q.loc[f, 'below_target']), page1_clicks_below_break_even=int(q.loc[f, 'clicks_below_break_even']), page1_clicks_below_target=int(q.loc[f, 'clicks_below_target']))
                rec.update({f'p_iter{it}': it_p.loc[f, it] if it in it_p.columns and f in it_p.index else np.nan for it in range(1, N_ITER + 1)})
                rec.update(share_E1=gf['e1'].mean(), share_E1_reachable=gf['e1_reach'].mean(), share_E2=gf['e2'].mean(), share_E2_reachable=gf['e2_reach'].mean(), share_EB1=gf['eB1'].mean(), mean_pages=gf.groupby('order').size().mean(), share_seeds_already_found=gf.loc[gf['iteration'] >= 2, 'seed_already_found'].mean())
                for acc in ('page_sum', 'session', 'session_seed'):
                    mm = MIN[acc]
                    row = mm[(mm['arm'] == arm) & (mm['variant'] == variant) & (mm['K'] == k) & (mm['file_name'] == f)].iloc[0]
                    rec.update({f'min_saved_{acc}_{rule}': row[rule] for rule in ('all', 'oracle', 'observable')})
                per_roi.append(rec)
PER_ROI_OUT = pd.DataFrame(per_roi)
PER_ROI_OUT.to_csv(OUT['per_roi'], index=False)
TARGETS.to_csv(OUT['targets'], index=False)
GRIDT.to_csv(OUT['grid'], index=False)
SUMMARY_OUT = pd.concat(SUMMARY, ignore_index=True)
lead = ['table', 'metric', 'arm', 'variant', 'K', 'iteration', 'bin', 'domain', 'group', 'contrast', 'accounting', 'mean', 'ci_lo', 'ci_hi', 'n_roi']
SUMMARY_OUT = SUMMARY_OUT[[c for c in lead if c in SUMMARY_OUT.columns] + [c for c in SUMMARY_OUT.columns if c not in lead]]
SUMMARY_OUT.to_csv(OUT['summary'], index=False)
for name, p in OUT.items():
    print(f'-> {p} ({os.path.getsize(p) / 1e6:.1f} MB)')
print(f'\nnotebook total {(time.time() - NB_T0) / 60:.1f} min')

-> ../../results/iteration_breakeven_candrefine_rerank_mo_pad61_select_pages.csv (29.6 MB)
-> ../../results/iteration_breakeven_candrefine_rerank_mo_pad61_select_per_roi.csv (0.5 MB)
-> ../../results/iteration_breakeven_candrefine_rerank_mo_pad61_select_targets.csv (0.1 MB)
-> ../../results/iteration_breakeven_candrefine_rerank_mo_pad61_select_grid.csv (0.1 MB)
-> ../../results/iteration_breakeven_candrefine_rerank_mo_pad61_select_summary.csv (1.4 MB)

notebook total 13.1 min


## Verdict and adversarial-check log

Every `C1` number below is **in-sample**. Its rule was found on these 49 ROIs, and NOTEBOOK chose it on these same
294 clicks. The comparison arms carry no selection.

### The answer

**The target scales with the number of unfound mitoses r.** At the central setting (v = 2 s, T = 10 min), the target
precision of a full page is 2p\* = 0.009·r at K = 10, 0.0078·r at K = 20 and 0.0074·r at K = 30:

| r | K = 10 | K = 20 | K = 30 |
|---|---|---|---|
| 10 | 0.09 | 0.08 | 0.07 |
| 20 | 0.18 | 0.16 | 0.15 |
| 40 | 0.36 | 0.31 | 0.30 |
| 80 | 0.72 | 0.63 | 0.60 |
| 160 | unreachable | unreachable | unreachable |

No page can reach E ≥ 2 above r ≈ 111, 128 and 134 for K = 10, 20 and 30.

**Page 1 reaches the target, and the share falls with every page.** `C1`'s share of pages with E ≥ 2 first drops
below half:
- on page 4 at K = 10 (0.73, 0.60, 0.56, 0.46);
- on page 3 at K = 20 (0.73, 0.54, 0.40);
- on page 2 at K = 30 (0.73, 0.43).
- **Page 1, central, K = 10:**
  - `C1` reaches E ≥ 2 on 72 % of first pages, and on 81 % of those where E ≥ 2 is reachable.
  - Production `tm_unfiltered` reaches 56 % and 63 %.
  - 8 of the 44 reachable ROIs miss the target with `C1`. `211.tiff` scores 0 on every arm, a generator miss.
- **Later pages:** by page 6 the share falls to 27–31 % at K = 10 and to 11–19 % at K = 20 and 30, for every arm.
- **Time saved:** a 6-page session saves `C1` a median of 2.0 min per ROI (K = 10), against 1.5 min for production.
  - The paired gain over production is +0.51 [0.34, 0.66] min per ROI.
  - `C1`'s re-rank alone is `C1 − od` = +0.30 [0.18, 0.41]. The other ~40 % comes from the refusal filter and from `od`
    beating `tm`.

**Everything hinges on the unmeasured pass time T.**
- At T = 5 min every arm loses time at K ≥ 20. At T = 20 min every arm saves 3–8 min.
- The break-even pass time is 5.8 min [IQR 4.5, 8.0] for a K = 10 session with `C1`, and 6.4 min [5.3, 9.5] for
  production. **Timing one real manual pass would settle the verdict.**

### Key tables (central: v 2 s, T 10 min, c 5 s, L 2 s)

**Share of pages reaching the target, K = 10.** Each cell is all pages / reachable pages only.

| page | `C1` E≥1 | `C1` E≥2 | `tm_unfiltered` E≥2 | `od_unfiltered` E≥2 | `od` E≥2 |
|---|---|---|---|---|---|
| 1 | 0.90 / 0.92 | 0.73 / 0.81 | 0.57 / 0.63 | 0.61 / 0.68 | 0.65 / 0.72 |
| 2 | 0.88 / 0.90 | 0.60 / 0.66 | 0.56 / 0.61 | 0.57 / 0.62 | 0.55 / 0.60 |
| 3 | 0.81 | 0.56 / 0.59 | 0.47 / 0.50 | 0.44 / 0.47 | 0.42 / 0.45 |
| 4 | 0.70 | 0.46 / 0.49 | 0.37 / 0.40 | 0.42 / 0.45 | 0.43 / 0.46 |
| 5 | 0.62 | 0.37 / 0.39 | 0.32 / 0.34 | 0.37 / 0.39 | 0.35 / 0.37 |
| 6 | 0.56 | 0.29 / 0.31 | 0.29 / 0.30 | 0.31 / 0.33 | 0.27 / 0.29 |

These are simulated pages, so page 1 is weighted by how often each click comes first. The exact page-1 values come
from the standalone pages:

| arm | E≥2, all | E≥2, reachable |
|---|---|---|
| `C1` | 0.72 | 0.81 |
| `tm_unfiltered` | 0.56 | 0.63 |
| `od_unfiltered` | 0.61 | 0.68 |
| `od` | 0.64 | 0.72 |

**`C1`, K = 10, share of pages with E ≥ 2 by density bin** (all / reachable only):

| R0 | page 1 | page 2 | page 3 | page 4 | page 5 | page 6 |
|---|---|---|---|---|---|---|
| ≤ 20 | 0.92 | 0.76 | 0.69 | 0.50 | 0.36 | 0.26 |
| 21–40 | 0.85 | 0.66 | 0.48 | 0.38 | 0.30 | 0.24 |
| 41–80 | 0.90 | 0.73 | 0.68 | 0.59 | 0.40 | 0.39 |
| > 80 | 0.23 / 0.39 | 0.24 / 0.36 | 0.36 / 0.48 | 0.39 / 0.52 | 0.41 / 0.55 | 0.30 / 0.40 |

The other bins have every page reachable.

**Pooled over all 6 pages, E ≥ 2 on reachable pages:**

| arm | K = 10 | K = 20 | K = 30 |
|---|---|---|---|
| `C1` | 0.54 | 0.40 | 0.31 |
| `od_unfiltered` | 0.49 | 0.37 | 0.30 |
| `tm_unfiltered` | 0.46 | 0.37 | 0.30 |
| `od` | 0.48 | 0.37 | 0.29 |

**Page 1 per ROI below target, central.** The ROI's mean P@K over its 6 clicks is compared with 2p\*.
- **`C1`, K = 10:** 13 of 49 ROIs miss the target.
  - 5 of them cannot reach E ≥ 2: `246`, `459`, `300`, `301` and `548`, with R0 116–239.
  - 8 of the 44 reachable ROIs miss: `211` (p = 0.00), `242`, `546`, `220`, `291`, `245`, `293` and `249`.
  - 4 ROIs are below break-even: `211`, `245`, `301` and `548`.
- **The comparison arms**, below target among reachable ROIs:
  - `od_unfiltered` 15 of 44;
  - `tm_unfiltered` 16 of 44;
  - `od` 13 of 44.

**Minutes saved per ROI, all 6 pages, central.** Each cell is the median, the IQR, and the share of ROIs with a net
saving.

| K | accounting | `C1` | `od_unfiltered` | `tm_unfiltered` | `od` |
|---|---|---|---|---|---|
| 10 | **session (headline)** | 2.0 [0.7, 3.3] 88 % | 1.4 [0.2, 3.0] 84 % | 1.5 [0.1, 2.4] 82 % | 1.5 [0.3, 3.0] 86 % |
| 20 | session | 1.2 [0.1, 2.2] 76 % | 0.9 | 0.8 | 1.0 |
| 30 | session | 0.1 [−0.9, 1.4] 57 % | −0.1 | −0.1 | −0.0 |
| 10 | page sum (the prompt's, inflated) | 3.6 [1.6, 6.6] 90 % | 2.6 | 2.8 | 2.8 |
| 10 | session minus seed premium | 1.3 [−0.1, 2.3] 73 % | 0.9 | 1.0 | 0.9 |

**Stopping, `C1`, session accounting.** At K = 10 the observable stop (after the first zero-TP page) saves 2.0 min,
close to the oracle's 2.2. At K = 30 it saves only 1.0 min against the oracle's 2.3, and all 6 pages save 0.1.

**`C1` minus production, paired over ROIs, session accounting, all pages:**

| K | minutes | W/L/T |
|---|---|---|
| 10 | +0.51 [0.34, 0.66] | 40/8/1 |
| 20 | +0.39 [0.27, 0.51] | |
| 30 | +0.23 [0.14, 0.32] | |

`C1` minus `od_unfiltered` is +0.31 [0.17, 0.44] min at K = 10. `C1` minus `od` is +0.30 [0.18, 0.41]; that is the
re-rank alone.

Per session at K = 10, `C1` confirms +2.2 [1.4, 2.9] more mitoses than production. Against `od`, the re-rank alone, it
confirms +1.6 [1.1, 2.2] more.

**Across the grid (`C1`, K = 10):**

| quantity | range over the 9 settings |
|---|---|
| page-1 E ≥ 2 share | 0.25 (v 3 s, T 5 min) to 0.97 (v 1.5 s, T 20 min) |
| ROIs below target at page 1 | 36 to 1 |
| median minutes saved per ROI | −1.4 to +7.1 |

`C1` has the highest page-1 E ≥ 2 share in 24 of 27 cells (K × setting). In the other 3, all at T = 5 min, it ties
or trails `od` or `od_unfiltered` by at most 0.004. Over all 6 pages it leads at K = 10 in every setting. At K = 20 and
30 another arm leads in 8 of 18 cells, by at most 0.03.

**Regime B** (a full pass is mandatory; v = 2 s):
- `p*_B` is 0.68, 0.45 or 0.27 for `c_saved` = 4, 6 or 10 s.
- `C1`'s page-1 share with E_B ≥ 1 is 0.50, 0.76 or 0.92, and the share over all pages is 0.25, 0.40 or 0.54.

### Caveats

- **In-sample.** `C1` was chosen on these clicks, and its rule's terms were chosen on these ROIs. Its page-1 gain over
  `od` is +4.86 [+1.26, +8.67] pts at P@10 (bootstrap added here), and that interval is optimistic too. No held-out set
  exists yet.
- **Unmeasured parameters.**
  - T dominates every conclusion.
  - v is assumed equal for true and false candidates, and for look-alikes.
  - The pathologist is assumed to judge perfectly against the MIDOG++ consensus. "Unlabelled" hits count as false.
- **Pool cap.**
  - Page-1 coverage is only 0.48 on average per ROI. The ROI means are 24 reachable mitoses out of 56 unfound, that
    is, within 7.5 µm of a pool candidate. At K = 10 the
    method's own ceiling is 0.90 on page 1 and 0.40 by page 6.
  - Refilling from the deep list cannot be tested for `C1`, because the deep rows have no features.
- **Fixed clicks.** The 6 recorded clicks are reused as seeds, and by page 6, 46–71 % of them were already found by
  the tool. A real user would box an unfound mitosis, which may give a fresher pool. Late pages here may therefore be
  pessimistic.
- **Regime.** Regime A presumes a partial-yield goal. For an exhaustive count a full pass is mandatory, and regime B's
  targets apply instead: 0.27–0.68 at K = 10.

### Adversarial-check log

1. **The model, re-derived.**
   - p\* = r·(nv+c+L)/(nT) and E = TP·T/(r·cost) follow from a manual yield of r/T. The units check out, as mitoses per
     candidate. p\* ∝ r holds only because T is fixed.
   - Saved seconds = (E−1)·cost is exact per page, and so are E·p\* = p and E_max = min(n, r)·T/(r·cost). All were
     asserted on 9000 random pages.
   - The prompt's "about 111" is right: 111.1 at K = 10, 127.7 at K = 20 and 134.3 at K = 30.
   - **Flag (prompt).** Summing (E−1)·cost over pages is not "time saved against no tool". Each later page is valued
     at T/r_j > T/R0.
     - Two pages that find 5 and then 5 of 10 mitoses are valued at 1.5·T. One hand pass finds all 10 in T.
     - Per-page E is kept as the marginal stopping test, which is correct. The headline minutes use the session
       accounting ΣTP·T/R0 − Σcost.
     - The page sum overstates the median saving at K = 10 by 1.6 min (3.6 against 2.0).
   - **Flag (prompt).** Seed finding is neutral only at the marginal rate. In the session view, hand-finding a later
     seed in a depleted ROI costs extra. That lowers the K = 10 median to 1.3 min.
   - **The box.** Charging c in full is slightly conservative, since a manual pass would also mark that mitosis. Nothing
     is double counted.
   - **Rate against order statistic.** Finding m of r by a systematic scan takes m·T/(r+1), not m·T/r. That is a 2×
     difference at r = 1 and 1.1× at r = 10. Not applied.
   - **T, area and density.**
     - ROI areas span 1.988–2.020 mm², so a fixed T per ROI is not an area artefact.
     - T fixed against density is questionable. With T_eff = T + r·c_saved, `p*_A` is bounded by `p*_B`, and nothing is
       unreachable. `C1`'s page-1 E ≥ 2 share is then 0.94 and its all-pages share 0.70.
   - **Added.** A closed-form break-even T per ROI.
2. **NOTEBOOK's numbers, recomputed from its CSVs.** Everything reproduces:
   - all 11 arms' pooled P@K;
   - `per_roi`;
   - all 54 `delta_stats` rows, pooled delta and W/L/T;
   - the spec's `all_roi_scores` (exact fractions);
   - every leave-one-domain-out fold's F5 direction, pick and held-out delta;
   - the pooled AUCs;
   - the refusal table;
   - the spec's SHA-1.

   The quoted P@10 values reproduce: 0.637, 0.589, 0.563 and 0.542. `C1` leads at every K on integer TP, by 39, 36 and 81
   TP, with no ties.

   Flags on NOTEBOOK:
   - (a) Every fold picked `C1` = REF, so every held-out delta is 0 by construction. The leave-one-domain-out is not
     held-out evidence that `C1` beats `od`.
   - (b) `C1 − od_unfiltered` (+7.48 pts at P@10) mixes the refusal filter (+2.62) with the re-rank (+4.86).
     - The refusals are mostly ROI-border candidates: 653 rows, 534 of them unlabelled.
     - In the product, whole-slide context would not refuse them, so part of +2.62 is a benchmark artefact.
   - (c) No CIs are reported. They have been added here.
   - (d) Two pre-declared directions are contradicted by the data:
     - F7 `spiky` has a pooled AUC of 0.426 in its declared direction, above 0.5 on only 12 of 49 ROIs;
     - F9 has a pooled AUC of 0.483.

     C2, C4 and C5 carry these terms. "C1 wins" partly reflects mis-specified alternatives, and is not evidence that
     those cues are useless.
3. **Selection bias.**
   - `C1` is in-sample twice: its terms were chosen on these ROIs earlier, and it was picked here. Every `C1` figure is
     labelled.
   - Its session-level advantage over production is small: +0.5 min per ROI at K = 10.
4. **Why later pages fall.** The click at page j is a uniformly random one of 6, so its standalone precision equals
   page 1's. This is verified: `p_fresh` stays at 0.64 at K = 10. The whole decline therefore comes from the session's
   removals.
   - **Factors, `C1` at K = 10 on page 6 against page 1:**

     | factor | ratio |
     |---|---|
     | r | 0.58 |
     | mitoses reachable from the remaining saved pool | 0.28 (coverage 0.41) |
     | ceiling | 0.44 |
     | ranking efficiency | 0.79 |
     | precision | 0.34 |

     Saved-pool coverage falls faster than depletion. At K = 20 and 30, efficiency ratios above 1 come from the
     ceiling binding, not from better ranking.
   - **Refill test.**
     - **Flag (prompt).** "The saved pools cannot model that" is partly wrong. NOTEBOOK's input
       `deep_pool_mo_pad61_49roi_6seed_postD11_top1000.csv` begins with each saved pool, so refill can be modelled for
       `od_unfiltered` and `tm_unfiltered`.
     - Refill raises coverage: 0.21 → 0.28 at page 6 for K = 10, and 0.07–0.08 → 0.16–0.19 for K = 30.
     - It does not raise precision. For a tm-ranked page this is a logical necessity: refill cannot change the top K by
       score until the saved pool under-fills.
     - For `od` the extra deep candidates add little. At K = 30 late pages they lower precision (0.089 → 0.066) and cost
       0.3–0.4 min per ROI.
     - So the late decline is not a cap artefact for these rankers. The remaining mitoses sit below hundreds of
       better-scored false positives, which is consistent with the deep-pool memory ("deep and diluted").
   - **`C1`'s late pages** are slightly below the comparison arms at K = 20 and 30. That is the mirror of its early
     harvest from the same pools: its cumulative mitoses lead after every page. It is not degradation.
5. **Matching across pages.**
   - No annotation is ever matched twice; this is asserted on every page.
   - Exclusions hide an unfound mitosis on about 3 % of pages from page 3 on. Nearly all of those mitoses lie within R
     of a shown candidate, so the user did see something there.
   - A candidate on the current seed stays and counts as a false positive. A candidate on an earlier seed is removed.
     This keeps NOTEBOOK's page-1 convention, but it is asymmetric.
6. **Robustness.**
   - `C1`'s page-1 lead holds in 24 of 27 grid cells; in 3 cells at T = 5 min it ties or trails by at most 0.004. It
     also holds in regime B and in the density-T variant, where `C1` is the sole leader in all 27 and all 9 cells
     (checked in code on exact values). Over all 6 pages the lead holds at K = 10, but not at K = 20
     and 30 (8 of 18 cells, by at most 0.03).
   - "Reaching the target" does not survive the grid: it flips with T, and at K ≥ 20 also with v.
   - Recomputed against full-pool order for `C1`: |Δp| ≤ 0.0032 per iteration. The minutes difference at K = 10 is
     −0.017 [−0.042, 0.008]. 2 of 15 cells have CIs at the edge of 0, consistent with chance.
7. **Sanity anchors.**
   - Iteration 1 equals NOTEBOOK's TP@10/20/30 exactly on all 294 standalone pages, for every arm and variant (gate 3).
   - The simulated iteration-1 ROI mean differs from it by at most +0.0044, because of order weighting.
   - Sparse ROIs fall hardest: R0 ≤ 20 goes from 0.49 to 0.035 at K = 10.
8. **Independent re-implementation.** A separate script (in the scratchpad, not in the repo) reproduced all 69 844
   primary pages exactly on r, n, TP, look-alikes, `seed_already_found` and removals. It used its own JSON parse,
   brute-force matcher and rank arithmetic. It was run on the first run's output and again on the final `pages.csv`.
9. **Gates, all passed.**
   - The four copied ranking functions' source SHA-1 equals the spec's, and the spec file's SHA-1 equals the one
     NOTEBOOK printed.
   - Gate 1: all 294 pools match NOTEBOOK's `n_pool` and `n_accepted`, with 6 distinct clicks per ROI.
   - Gate 2: 2646 stored position columns are reproduced exactly by NOTEBOOK's own `arm_lists`, and the simulator's
     full orders equal them.
   - Gate 3: 6174 standalone first pages equal NOTEBOOK's TP@K, including the refill variant.
   - Gate 4: the JSON category-1 count equals `n_gt_mitotic + 1` on every click, and every click is category 1.
     `image_id` matches the JSON.
   - Gate 5: every invariant held on all 122 245 pages, and p\* and E are finite on every shown page.
   - Gate 6: 6 hand-derived toy sessions reproduce, the refill one included.
   - Every deep list begins exactly with its saved pool.
   - The ROI areas are equal to within 1.6 %.
10. **What failed on my side, and was fixed.**
   - One toy expectation was mis-derived (`n_removed_found`). I caught it on re-derivation before the first run.
   - The first run's per-bin E-share tables for `C1` did not filter by K, so all three K blocks showed pooled numbers.
   - W/L/T counts printed as floats.
   - **Deviation.** `pages.csv` stores `n`, `TP`, `r` and `R0` plus central-setting columns, at 10 significant digits.
     The grid is recomputed in the notebook rather than stored 9 times per page.